In [1]:
# 1.1 检查并安装必要的库
# 如果缺少某些库，可以取消下面相应行的注释来安装

# 安装核心数据处理库
# !pip install pandas numpy matplotlib seaborn -q

# 安装空间分析库(geopandas安装可能较复杂，建议使用conda)
# 如果使用pip安装遇到问题，建议使用：conda install geopandas
# !pip install geopandas -q

# 安装可视化相关库
# !pip install contextily folium -q

# 安装机器学习/统计库
# !pip install scikit-learn -q

# 安装进度条
# !pip install tqdm -q

# 1.2 导入所有必要的库
import os
import sys
import json
import time
import math
import hashlib
import pickle
import warnings
from datetime import datetime
from pathlib import Path
from functools import lru_cache
from collections import defaultdict, OrderedDict

import pandas as pd
import numpy as np
import requests
from tqdm.notebook import tqdm  # Jupyter专用进度条

# 空间分析库
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle, PathPatch, FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.font_manager import FontProperties
from matplotlib import patheffects
import matplotlib.patches as mpatches
from shapely.geometry import Point, Polygon, LineString, box
import contextily as ctx

# 机器学习/统计分析
from sklearn.preprocessing import MinMaxScaler

# 设置中文字体和警告
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'KaiTi']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

print("√ 所有必要的库已导入")

√ 所有必要的库已导入


In [2]:
# 2.1 配置类（关键词搜索版）
class Config:
    """统一配置管理类（仅关键词搜索）"""
    def __init__(self):
        # 高德地图API配置
        # 重要：请替换为您的实际API密钥
        self.AMAP_KEY = ""  # 请务必更换为您自己的密钥
        
        # 路径配置
        # 获取当前工作目录
        self.base_dir = Path.cwd()
        
        # 子目录配置
        self.data_dir = self.base_dir / "data"
        self.output_dir = self.base_dir / "results"
        self.cache_dir = self.base_dir / "cache"
        self.figures_dir = self.output_dir / "figures"
        self.reports_dir = self.output_dir / "reports"
        
        # 创建所有必要的目录
        self._create_directories()
        
        # 研究区域配置
        self.target_city = "惠州市"
        self.target_district = "惠城区"
        
        # 惠城区大致坐标范围
        self.bounds = {
            'min_lon': 114.3,
            'max_lon': 114.6,
            'min_lat': 22.9,
            'max_lat': 23.2
        }
        
        # API配置
        self.api_config = {
            'base_url': "https://restapi.amap.com/v3/place/text",
            'polygon_url': "https://restapi.amap.com/v3/place/polygon",
            'page_size': 20,      # 每页数据量
            'max_pages': 10,      # 最大页数
            'request_delay': 0.5, # 请求延迟(秒)
            'max_retries': 3,     # 最大重试次数
            'timeout': 30         # 超时时间
        }
        
        # 缓冲区配置
        self.buffer_distance = 1000  # 服务区半径(米)
        self.road_buffer_width = 20  # 道路缓冲宽度(米)
        
        # POI关键词配置（仅使用关键词搜索）
        self.default_poi_categories = {
            '教育设施': ['幼儿园', '小学', '中学', '大学', '培训机构'],
            '医疗设施': ['医院', '诊所', '社区卫生服务中心', '药店'],
            '商业设施': ['超市', '便利店', '菜市场', '商场', '银行', '餐饮'],
            '文体设施': ['公园', '体育场馆', '图书馆', '文化馆', '电影院'],
            '行政服务': ['政府机关', '派出所', '社区服务中心', '邮政'],
            '交通设施': ['公交车站', '地铁站', '停车场'],
            '生活服务': ['理发店', '维修店', '洗衣店', '美容店']
        }
        
        # CRITIC法参数配置
        self.critic_config = {
            'std_weight': 0.5,  # 标准差权重
            'corr_weight': 0.5,  # 相关性权重
        }
        
        # 维度权重（将基于CRITIC法动态计算）
        self.dimension_weights = {
            'supply': 0.40,   # 供给维度
            'diversity': 0.35,  # 多样性维度
            'access': 0.25     # 可达性维度
        }
        
        # 用户数据路径配置（请根据实际情况修改）
        self.user_file_paths = {
            'boundary': r"C:\Users\33353\Desktop\作业\441302.shp",  # 行政区边界
            'road_network': r"D:\惠城区道路路网_441302_Shapefile_(poi86.com)\441302.shp",
            'poi_local': r"D:/Jupyter notebook 代码/data/collected_poi.geojson",  # 本地POI数据
            'community': r"D:/Jupyter notebook 代码/惠城区小区数据/惠城区居民小区_20260317_043627.csv"
        }
        
        print("√配置初始化完成（关键词搜索版）")
        print(f"工作目录：{self.base_dir}")
        print(f"数据目录：{self.data_dir}")
        print(f"输出目录：{self.output_dir}")
    
    def _create_directories(self):
        """创建所有必要的目录"""
        directories = [
            self.data_dir,
            self.output_dir,
            self.cache_dir,
            self.figures_dir,
            self.reports_dir
        ]
        for directory in directories:
            directory.mkdir(parents=True, exist_ok=True)
    
    def update_paths(self, **kwargs):
        """动态更新路径配置"""
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, Path(value))
                print(f"√更新{key}: {value}")
                
config = Config()  # 这行代码会读取您修改后的配置，并创建名为 config 的实例

√配置初始化完成（关键词搜索版）
工作目录：D:\Jupyter notebook 代码
数据目录：D:\Jupyter notebook 代码\data
输出目录：D:\Jupyter notebook 代码\results


In [3]:
# 3.1 改进的内存缓存装饰器
def memory_cache(ttl=3600, maxsize=128):
    """内存缓存装饰器，使用有序字典实现LRU"""
    cache = OrderedDict()
    
    def decorator(func):
        def wrapper(*args, **kwargs):
            # 生成缓存键（排除self）
            key_items = [func.__name__]
            
            # 处理args，跳过self
            if args:
                key_items.append(str(args[1:]) if len(args) > 1 else str(args))
            
            if kwargs:
                key_items.append(str(sorted(kwargs.items())))
            
            key = hashlib.md5(":".join(key_items).encode()).hexdigest()
            
            # 检查缓存是否存在且未过期
            if key in cache:
                cached_time, result = cache[key]
                if time.time() - cached_time < ttl:
                    # 更新访问顺序
                    cache.move_to_end(key)
                    return result
                else:
                    # 缓存过期，删除
                    del cache[key]
            
            # 执行函数
            result = func(*args, **kwargs)
            
            # 更新缓存
            cache[key] = (time.time(), result)
            
            # 清理过期和超量缓存
            current_time = time.time()
            expired_keys = [k for k, (t, _) in cache.items() if current_time - t >= ttl]
            for k in expired_keys:
                del cache[k]
            
            if len(cache) > maxsize:
                # 移除最久未使用的
                cache.popitem(last=False)
            
            return result
        return wrapper
    return decorator

# 3.2 POI数据采集器（关键词搜索版）
class POICollector:
    """POI数据采集器-仅使用关键词搜索，集成缓存和频率控制"""
    
    def __init__(self, config):
        self.config = config
        self.session = requests.Session()  # 使用Session提高连接复用
        self.request_timestamps = []  # 存储最近请求的时间戳，用于频率控制
        
    def _control_request_rate(self):
        """
        控制请求频率，确保不超过每秒3次
        移除超过1秒的时间戳，如果当前时间戳数量>=3则等待
        """
        current_time = time.time()
        # 移除1秒之前的记录
        self.request_timestamps = [ts for ts in self.request_timestamps 
                                  if current_time - ts < 1.0]
        
        # 如果最近1秒内已有3次或更多请求，则等待
        while len(self.request_timestamps) >= 3:
            # 计算需要等待的时间（直到最早的请求超过1秒前）
            oldest_time = self.request_timestamps[0]
            wait_time = 1.0 - (current_time - oldest_time)
            if wait_time > 0:
                time.sleep(wait_time + 0.01)  # 加一点缓冲时间
                current_time = time.time()
                # 重新清理时间戳列表
                self.request_timestamps = [ts for ts in self.request_timestamps 
                                          if current_time - ts < 1.0]
            else:
                break
    
    @memory_cache(ttl=24 * 3600)  # 缓存24小时
    def search_poi(self, keyword, city=None, page=1, use_cache=True):
        """
        搜索POI数据（关键词搜索）
        
        参数:
            keyword: 搜索关键词
            city: 城市名称，默认使用配置中的行政区
            page: 页码
            use_cache: 是否使用缓存
            
        返回:
            dict: API响应数据
        """
        if city is None:
            city = self.config.target_district
        
        params = {
            'key': self.config.AMAP_KEY,
            'keywords': keyword,  # 使用keywords参数
            'city': city,
            'citylimit': 'true',
            'offset': self.config.api_config['page_size'],
            'page': page,
            'extensions': 'base',  # 只获取基础信息，节省流量
            'output': 'json'
        }
        
        # 检查API密钥
        if self.config.AMAP_KEY == "您的高德地图API密钥":  # 检查是否为示例密钥
            print("!警告：请先在Config类中配置您自己的高德地图API密钥")
            return None
        
        # 发送请求（带频率控制和重试机制）
        for retry in range(self.config.api_config['max_retries'] + 1):
            try:
                # 1. 频率控制：确保不超过每秒3次
                self._control_request_rate()
                
                # 2. 记录当前请求时间
                request_start_time = time.time()
                
                if retry > 0:
                    wait_time = 2 ** retry
                    print(f"第{retry}次重试，等待{wait_time}秒...")
                    time.sleep(wait_time)
                
                response = self.session.get(
                    self.config.api_config['base_url'],
                    params=params,
                    timeout=self.config.api_config['timeout']
                )
                response.raise_for_status()
                
                # 3. 请求成功后添加时间戳记录
                self.request_timestamps.append(time.time())
                
                data = response.json()
                if data.get('status') == '1':
                    return data
                else:
                    error_msg = data.get('info', '未知错误')
                    print(f"API错误：{error_msg}")
                    if "访问已超出" in error_msg or "配额" in error_msg:
                        print("API额度不足，请检查配额")
                        break
                        
            except requests.exceptions.RequestException as e:
                if retry == self.config.api_config['max_retries']:
                    print(f"×请求失败：{e}")
                    break
                continue
        
        return None
    
    def batch_collect(self, keyword_dict=None, verbose=True, save_path=None):
        """
        批量采集POI数据（关键词搜索）
        
        参数:
            keyword_dict: 关键词字典，格式：{'类别': [关键词列表]}
                         如果为None，则使用config.default_poi_categories
            verbose: 是否显示详细进度
            save_path: 保存路径，如果不为None则采集后自动保存为CSV
            
        返回:
            GeoDataFrame: POI数据
        """
        if keyword_dict is None:
            keyword_dict = self.config.default_poi_categories
        
        all_pois = []
        stats = {
            'total_keywords': 0,
            'successful': 0,
            'failed': 0,
            'total_pois': 0
        }
        
        if verbose:
            print("=" * 60)
            print("开始批量采集POI数据（关键词搜索，频率控制：每秒≤3次请求）")
            print("=" * 60)
        
        # 清空时间戳记录，开始新的批量采集
        self.request_timestamps = []
        
        for category, keywords in keyword_dict.items():
            if verbose:
                print(f"\n采集类别：{category}")
                print(f"关键词：{keywords}")
            
            for keyword in keywords:
                stats['total_keywords'] += 1
                if verbose:
                    print(f"搜索：{keyword}")
                
                all_keyword_pois = []
                
                # 获取第一页
                first_page = self.search_poi(keyword, page=1)
                if not first_page:
                    stats['failed'] += 1
                    if verbose:
                        print(f"×获取失败")
                    continue
                
                # 解析第一页数据
                pois = first_page.get('pois', [])
                all_keyword_pois.extend(pois)
                
                total_count = int(first_page.get('count', 0))
                if verbose:
                    print(f"√找到{total_count}个POI")
                
                # 计算总页数
                if total_count > 0:
                    total_pages = min(
                        math.ceil(total_count / self.config.api_config['page_size']),
                        self.config.api_config['max_pages']
                    )
                    
                    # 获取剩余页面（如果有）
                    if total_pages > 1:
                        for page in tqdm(range(2, total_pages + 1),
                                        desc=f"获取{keyword}",
                                        leave=False,
                                        disable=not verbose):
                            # 这里会自动调用频率控制
                            page_data = self.search_poi(keyword, page=page, use_cache=False)
                            if page_data and page_data.get('status') == '1':
                                all_keyword_pois.extend(page_data.get('pois', []))
                
                # 添加类别信息
                for poi in all_keyword_pois:
                    poi['category'] = category
                    poi['sub_category'] = keyword
                
                all_pois.extend(all_keyword_pois)
                stats['successful'] += 1
                stats['total_pois'] += len(all_keyword_pois)
                
                if verbose:
                    print(f"√获取到{len(all_keyword_pois)}条记录")
                
                # 在关键词之间添加额外延迟，避免连续请求
                time.sleep(0.1)
        
        # 转换为DataFrame
        if all_pois:
            df = pd.DataFrame(all_pois)
            
            # 转换为GeoDataFrame
            geometries = []
            valid_pois = []
            for _, row in df.iterrows():
                location = row.get('location')
                if location and ',' in location:
                    lon, lat = map(float, location.split(','))
                    geometries.append(Point(lon, lat))
                    valid_pois.append(row)
                else:
                    # 如果没有坐标，跳过
                    continue
            
            if valid_pois:
                gdf = gpd.GeoDataFrame(valid_pois, geometry=geometries, crs="EPSG:4326")
                
                # 保存数据
                if save_path:
                    save_path = Path(save_path)
                    save_path.parent.mkdir(parents=True, exist_ok=True)
                    gdf.to_file(save_path, driver='GeoJSON', encoding='utf-8')
                    print(f"√POI数据已保存:{save_path}")
                
                if verbose:
                    print("\n" + "=" * 60)
                    print("☑批量采集完成")
                    print(f"成功关键词：{stats['successful']}/{stats['total_keywords']}")
                    print(f"总POI数量：{stats['total_pois']}")
                    print("=" * 60)
                
                return gdf
        
        if verbose:
            print("\n未获取到任何POI数据")
        return gpd.GeoDataFrame()
    
    # 保留原有的search_poi方法，用于向后兼容
    @memory_cache(ttl=24 * 3600)  # 缓存24小时
    def search_poi(self, keyword, city=None, page=1, use_cache=True):
        """
        搜索POI数据（关键词搜索，向后兼容）
        
        参数:
            keyword: 搜索关键词
            city: 城市名称，默认使用配置中的行政区
            page: 页码
            use_cache: 是否使用缓存
            
        返回:
            dict: API响应数据
        """
        if city is None:
            city = self.config.target_district
        
        params = {
            'key': self.config.AMAP_KEY,
            'keywords': keyword,  # 使用keywords参数
            'city': city,
            'citylimit': 'true',
            'offset': self.config.api_config['page_size'],
            'page': page,
            'extensions': 'base',
            'output': 'json'
        }
        
        # 检查API密钥
        if self.config.AMAP_KEY == "您的高德地图API密钥":
            print("!警告：请先在Config类中配置您自己的高德地图API密钥")
            return None
        
        # 发送请求（带频率控制和重试机制）
        for retry in range(self.config.api_config['max_retries'] + 1):
            try:
                self._control_request_rate()
                request_start_time = time.time()
                
                if retry > 0:
                    wait_time = 2 ** retry
                    print(f"第{retry}次重试，等待{wait_time}秒...")
                    time.sleep(wait_time)
                
                response = self.session.get(
                    self.config.api_config['base_url'],
                    params=params,
                    timeout=self.config.api_config['timeout']
                )
                response.raise_for_status()
                self.request_timestamps.append(time.time())
                
                data = response.json()
                if data.get('status') == '1':
                    return data
                else:
                    error_msg = data.get('info', '未知错误')
                    print(f"API错误：{error_msg}")
                    if "访问已超出" in error_msg or "配额" in error_msg:
                        print("API额度不足，请检查配额")
                        break
                        
            except requests.exceptions.RequestException as e:
                if retry == self.config.api_config['max_retries']:
                    print(f"×请求失败：{e}")
                    break
                continue
        
        return None

#3.3测试API连接
def test_api_connection():
    """测试API连接（关键词搜索版）"""
    print("测试高德地图API连接（关键词搜索）...")
    # 注意：这里的 `config` 变量需要在函数外部已定义
    # 通常在主流程中，会先执行 `config = Config()` 来初始化
    collector = POICollector(config)
    # 使用关键词“医院”进行测试，而非不存在的类型编码
    test_data = collector.search_poi("医院", page=1)
    if test_data and test_data.get('status') == '1':
        count = test_data.get('count', 0)
        print(f"√ API连接正常，找到约{count}个相关地点")
        return True
    else:
        print("✗ API连接失败，请检查API密钥和网络")
        return False
    
# 创建POI采集器实例
poi_collector = POICollector(config)

In [4]:
# 4. 增强数据处理器
class CSVCommunityProcessor:
    """
    增强型小区数据处理器
    支持CSV和Excel格式，自动检测编码和坐标列
    """
    
    def __init__(self, file_path):
        self.file_path = Path(file_path)
        self.df = None
        self.longitude_col = None
        self.latitude_col = None
    
    def load_and_validate(self):
        """
        加载并验证小区数据
        
        返回:
            DataFrame: 加载的数据框
        """
        print(f"加载数据文件: {self.file_path}")
        
        if not self.file_path.exists():
            raise FileNotFoundError(f"文件不存在: {self.file_path}")
        
        # 根据文件扩展名选择读取方式
        if self.file_path.suffix.lower() in ['.xlsx', '.xls']:
            self._load_excel()
        elif self.file_path.suffix.lower() == '.csv':
            self._load_csv()
        else:
            raise ValueError(f"不支持的文件格式: {self.file_path.suffix}")
        
        # 验证数据
        self._validate_data()
        return self.df
    
    def _load_excel(self):
        """加载Excel文件"""
        try:
            # 尝试openpyxl引擎
            self.df = pd.read_excel(self.file_path, engine='openpyxl')
            print(f"√ 使用openpyxl引擎成功读取Excel文件")
        except Exception as e1:
            print(f" openpyxl引擎失败: {e1}")
            # 尝试xlrd引擎（用于.xls文件）
            try:
                self.df = pd.read_excel(self.file_path, engine='xlrd')
                print(f"√ 使用xlrd引擎成功读取Excel文件")
            except Exception as e2:
                print(f" xlrd引擎也失败: {e2}")
                raise
    
    def _load_csv(self):
        """加载CSV文件，自动检测编码"""
        encodings = ['utf-8-sig', 'gbk', 'gb2312', 'latin1', 'utf-8']
        
        for encoding in encodings:
            try:
                self.df = pd.read_csv(
                    self.file_path,
                    encoding=encoding,
                    engine='python',  # 使用python引擎更稳定
                    on_bad_lines='skip'  # 跳过错误行
                )
                print(f"√ 使用编码 {encoding} 成功读取CSV文件")
                return
            except UnicodeDecodeError:
                continue
            except Exception as e:
                print(f"编码 {encoding} 读取失败: {e}")
                continue
        
        # 所有编码都失败
        raise ValueError(f"无法用任何编码读取CSV文件: {self.file_path}")
    
    def _validate_data(self):
        """验证加载的数据"""
        if self.df is None or self.df.empty:
            raise ValueError("数据框为空或加载失败")
        
        print(f"数据形状: {self.df.shape}")
        print(f"数据列名: {list(self.df.columns)}")
        
        # 检测坐标列
        self.longitude_col, self.latitude_col = self._detect_coordinate_columns()
        
        if not self.longitude_col or not self.latitude_col:
            print("! 无法自动检测经纬度列，将尝试默认列名")
            self._try_default_columns()
    
    def _detect_coordinate_columns(self):
        """
        智能检测经纬度列 - 修复版
        避免将cityname等列误判为纬度列
        """
        longitude_cols = []
        latitude_cols = []
        
        for col in self.df.columns:
            col_lower = str(col).lower().strip()
            
            # 经度列检测
            lon_patterns = ['lon', 'longitude', '经度', 'x', 'xcoord', 'lng', 'long']
            if any(pattern in col_lower for pattern in lon_patterns):
                # 额外验证：经度应该在合理范围内
                if pd.api.types.is_numeric_dtype(self.df[col]):
                    if self.df[col].between(110, 120, inclusive='both').any():
                        longitude_cols.append(col)
                        print(f"√ 识别为经度列: {col}")
            
            # 纬度列检测 - 关键修复：排除cityname等列
            lat_patterns = ['lat', 'latitude', '纬度', 'y', 'ycoord']
            if any(pattern in col_lower for pattern in lat_patterns):
                # 排除包含city、name等字样的列
                exclude_patterns = ['city', 'name', 'address', 'location', '城区']
                if not any(exclude in col_lower for exclude in exclude_patterns):
                    # 额外验证：纬度应该在合理范围内
                    if pd.api.types.is_numeric_dtype(self.df[col]):
                        if self.df[col].between(20, 30, inclusive='both').any():
                            latitude_cols.append(col)
                            print(f"√ 识别为纬度列: {col}")
        
        # 返回第一个匹配的列
        longitude = longitude_cols[0] if longitude_cols else None
        latitude = latitude_cols[0] if latitude_cols else None
        
        return longitude, latitude
    
    def _try_default_columns(self):
        """尝试默认列名"""
        default_lon_names = ['longitude', '经度', 'lon', 'lng', 'x']
        default_lat_names = ['latitude', '纬度', 'lat', 'y']
        
        for col in self.df.columns:
            if self.longitude_col is None and col in default_lon_names:
                self.longitude_col = col
                print(f"√ 使用默认经度列: {col}")
            if self.latitude_col is None and col in default_lat_names:
                self.latitude_col = col
                print(f"√ 使用默认纬度列: {col}")
        
        # 如果仍然没有找到，检查是否有数值型列可能是坐标
        if self.longitude_col is None or self.latitude_col is None:
            numeric_cols = self.df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) >= 2:
                # 假设前两个数值型列是经纬度
                self.longitude_col = numeric_cols[0]
                self.latitude_col = numeric_cols[1]
                print(f"√ 使用前两个数值型列作为坐标: {self.longitude_col}, {self.latitude_col}")
    
    def convert_to_geodataframe(self, crs="EPSG:4326"):
        """
        将数据框转换为GeoDataFrame
        
        参数:
            crs: 坐标系
            
        返回:
            GeoDataFrame: 包含几何列的地理数据框
        """
        if self.longitude_col is None or self.latitude_col is None:
            raise ValueError("未找到经纬度列，无法创建GeoDataFrame")
        
        # 检查坐标数据有效性
        valid_mask = (
            self.df[self.longitude_col].notna() &
            self.df[self.latitude_col].notna() &
            self.df[self.longitude_col].between(-180, 180) &
            self.df[self.latitude_col].between(-90, 90)
        )
        
        valid_count = valid_mask.sum()
        invalid_count = len(self.df) - valid_count
        
        if invalid_count > 0:
            print(f"移除 {invalid_count} 条无效坐标记录")
            self.df = self.df[valid_mask].copy()
        
        if valid_count == 0:
            raise ValueError("没有有效的坐标数据")
        
        # 创建几何列
        geometry = [
            Point(lon, lat)
            for lon, lat in zip(self.df[self.longitude_col], self.df[self.latitude_col])
        ]
        
        # 重命名列
        df_copy = self.df.copy()
        if self.longitude_col != 'longitude':
            df_copy = df_copy.rename(columns={self.longitude_col: 'longitude'})
        if self.latitude_col != 'latitude':
            df_copy = df_copy.rename(columns={self.latitude_col: 'latitude'})
        
        # 创建GeoDataFrame
        gdf = gpd.GeoDataFrame(df_copy, geometry=geometry, crs=crs)
        
        print(f"√ 成功创建GeoDataFrame: {len(gdf)} 个点")
        print(f"坐标系: {gdf.crs}")
        
        return gdf

# 测试增强处理器
def test_csv_processor():
    """测试增强处理器"""
    print("测试增强数据处理器...")
    # 这里可以添加测试代码
    # processor = CSVCommunityProcessor("your_file_path.csv")
    # df = processor.load_and_validate()
    # gdf = processor.convert_to_geodataframe()
    print("增强处理器类定义完成")

test_csv_processor()

测试增强数据处理器...
增强处理器类定义完成


In [5]:
class SpatialAnalyzer:
    """
    空间分析器 - 实现"缓冲区+路网限制"的核心研究方法
    修改：删除映射字典逻辑，直接使用CSV中的category和sub_category字段进行分类
    """
    def __init__(self, config):
        self.config = config
        self._crs_web_mercator = "EPSG:3857"  # Web墨卡托，用于准确距离计算
        self._crs_wgs84 = "EPSG:4326"         # WGS84，常用经纬度
        
    # 删除 _simplify_poi_type 方法，因为我们不再使用映射字典
    
    def load_and_unify_data(self, boundary_path, road_path, poi_path, community_path):
        """
        加载并统一所有空间数据的坐标系
        修改：优化POI数据加载，确保正确读取category和sub_category字段
        """
        print("=" * 60)
        print("步骤1: 加载并统一空间数据")
        print("-" * 60)
        
        try:
            # 1. 加载边界数据
            print("加载行政区边界...")
            boundary = gpd.read_file(boundary_path)
            print(f"  原始CRS: {boundary.crs}，要素数: {len(boundary)}")
            
            # 2. 加载路网数据
            print("加载路网数据...")
            roads = gpd.read_file(road_path)
            
            # 修复路网数据的CRS(如果缺失)
            if roads.crs is None:
                print("  ! 路网数据缺少CRS，尝试自动检测...")
                bounds = roads.total_bounds
                if (bounds[0] > -180 and bounds[2] < 180 and bounds[1] > -90 and bounds[3] < 90):
                    roads = roads.set_crs(self._crs_wgs84, allow_override=True)
                    print(f"  已设置CRS为: {self._crs_wgs84}")
                else:
                    roads = roads.set_crs("EPSG:4526", allow_override=True)
                    print(f"  已设置CRS为: EPSG:4526")
                    
            print(f"  原始CRS: {roads.crs}，线段数: {len(roads)}")
            
            # 3. 加载POI数据 - 修改：支持CSV格式并检查必要字段
            print("加载POI数据...")
            
            # 检查文件扩展名，如果是CSV格式则特殊处理
            if str(poi_path).lower().endswith('.csv'):
                print("  ! 检测到CSV格式的POI数据，使用pandas加载...")
                
                # 尝试多种编码读取CSV
                encodings = ['utf-8-sig', 'gbk', 'utf-8', 'latin1']
                df_poi = None
                
                for encoding in encodings:
                    try:
                        df_poi = pd.read_csv(poi_path, encoding=encoding)
                        print(f"    使用编码 {encoding} 成功读取CSV文件")
                        break
                    except Exception as e:
                        continue
                
                if df_poi is None:
                    raise ValueError(f"无法读取CSV文件: {poi_path}，尝试了多种编码均失败")
                
                # 检查必要的列
                required_cols = ['longitude', 'latitude', 'category']
                missing_cols = [col for col in required_cols if col not in df_poi.columns]
                
                if missing_cols:
                    raise ValueError(f"POI CSV文件缺少必要列: {missing_cols}。现有列: {list(df_poi.columns)}")
                
                # 创建几何列
                geometry = [Point(lon, lat) for lon, lat in 
                          zip(df_poi['longitude'], df_poi['latitude'])]
                
                # 创建GeoDataFrame
                poi = gpd.GeoDataFrame(
                    df_poi,
                    geometry=geometry,
                    crs=self._crs_wgs84
                )
                
                print(f"  √ 从CSV创建GeoDataFrame成功，包含 {len(poi)} 个POI点")
                print(f"  检测到的分类字段: category={list(poi['category'].unique())[:5]}...")
                if 'sub_category' in poi.columns:
                    print(f"               sub_category={list(poi['sub_category'].dropna().unique())[:5]}...")
            else:
                # 如果不是CSV，按原始方式加载
                poi = gpd.read_file(poi_path)
            
            print(f"  POI数据CRS: {poi.crs}，点数: {len(poi)}")
            
            # 4. 加载小区数据(使用增强处理器)
            print("加载小区数据...")
            try:
                # 尝试使用增强处理器
                processor = CSVCommunityProcessor(community_path)
                communities_df = processor.load_and_validate()
                communities = processor.convert_to_geodataframe()
                print(f"  ☑ 使用增强处理器成功加载")
            except Exception as e:
                print(f"  ! 增强处理器加载失败: {e}")
                print("  尝试回退到基本加载方式...")
                
                # 回退到基本加载方式
                if str(community_path).endswith(('.xlsx', '.xls')):
                    communities_df = pd.read_excel(community_path, engine='openpyxl')
                elif str(community_path).endswith('.csv'):
                    # 尝试不同编码
                    encodings = ['utf-8-sig', 'gbk', 'gb2312', 'latin1', 'utf-8']
                    for encoding in encodings:
                        try:
                            communities_df = pd.read_csv(community_path, encoding=encoding)
                            print(f"  使用编码{encoding}成功读取")
                            break
                        except:
                            continue
                    else:
                        raise ValueError(f"无法读取CSV文件: {community_path}")
                else:
                    raise ValueError(f"不支持的文件格式: {community_path}")
                    
                # 手动检测经纬度列
                longitude_col = None
                latitude_col = None
                
                for col in communities_df.columns:
                    col_lower = str(col).lower()
                    if longitude_col is None and any(pattern in col_lower for pattern in ['lon', 'longitude', '经度', 'x', 'lng']):
                        longitude_col = col
                    if latitude_col is None and any(pattern in col_lower for pattern in ['lat', 'latitude', '纬度', 'y']):
                        if 'city' not in col_lower and 'name' not in col_lower:
                            latitude_col = col
                            
                if longitude_col is None or latitude_col is None:
                    # 尝试默认列名
                    if 'longitude' in communities_df.columns and 'latitude' in communities_df.columns:
                        longitude_col, latitude_col = 'longitude', 'latitude'
                    elif '经度' in communities_df.columns and '纬度' in communities_df.columns:
                        longitude_col, latitude_col = '经度', '纬度'
                    else:
                        raise ValueError(f"无法识别经纬度列。现有列名: {list(communities_df.columns)}")
                
                # 重命名列
                communities_df = communities_df.rename(columns={
                    longitude_col: 'longitude',
                    latitude_col: 'latitude'
                })
                
                # 创建几何
                geometry = [Point(lon, lat) for lon, lat in 
                          zip(communities_df['longitude'], communities_df['latitude'])]
                communities = gpd.GeoDataFrame(
                    communities_df,
                    geometry=geometry,
                    crs=self._crs_wgs84
                )
                print(f"  √ 创建成功: {len(communities)}个小区点")
            
            # 数据预览
            print("\n数据预览(前3行):")
            preview_cols = []
            if 'name' in communities.columns:
                preview_cols.append('name')
            preview_cols.extend(['longitude', 'latitude'])
            
            if len(communities) > 0:
                print(communities[preview_cols].head(3).to_string())
            
            # 检查坐标范围
            if len(communities) > 0:
                lon_min, lon_max = communities['longitude'].min(), communities['longitude'].max()
                lat_min, lat_max = communities['latitude'].min(), communities['latitude'].max()
                print(f"  坐标范围: 经度({lon_min:.4f}~{lon_max:.4f})，纬度({lat_min:.4f}~{lat_max:.4f})")
            
            # 5. 统一坐标系到Web墨卡托(用于准确的距离计算)
            print(f"\n  统一所有数据到{self._crs_web_mercator}...")
            boundary_3857 = boundary.to_crs(self._crs_web_mercator)
            roads_3857 = roads.to_crs(self._crs_web_mercator)
            poi_3857 = poi.to_crs(self._crs_web_mercator)
            communities_3857 = communities.to_crs(self._crs_web_mercator)
            
            print("  ☑ 坐标系统一完成")
            print(f"  边界要素: {len(boundary_3857)}")
            print(f"  路网线段: {len(roads_3857)}")
            print(f"  POI点数: {len(poi_3857)}")
            print(f"  小区点数: {len(communities_3857)}")
            
            return boundary_3857, roads_3857, poi_3857, communities_3857
            
        except Exception as e:
            print(f"  X 数据加载失败: {e}")
            import traceback
            traceback.print_exc()
            return None, None, None, None
            
    def generate_road_network_service_areas(self, communities_gdf, roads_gdf, 
                                           distance_meters=1000, verbose=True):
        """
        核心方法: 基于真实路网生成服务区
        """
        print("\n" + "=" * 60)
        print("步骤2: 基于路网生成服务区(缓冲区+路网限制)")
        print(f"服务半径: {distance_meters}米")
        print("-" * 60)
        
        start_time = time.time()
        
        try:
            # 预处理路网: 创建道路缓冲区(模拟道路通行区域)
            if verbose:
                print("预处理路网数据...")
                
            # 将路网合并为单一几何对象(提高性能)
            roads_union = roads_gdf.geometry.unary_union
            
            # 创建道路缓冲区(道路宽度)
            roads_buffered = roads_union.buffer(self.config.road_buffer_width)
            
            service_areas_list = []
            failed_count = 0
            
            # 为每个小区生成服务区
            if verbose:
                iterator = tqdm(communities_gdf.iterrows(),
                              total=len(communities_gdf),
                              desc="生成服务区")
            else:
                iterator = communities_gdf.iterrows()
                
            for idx, community in iterator:
                try:
                    center_point = community.geometry
                    
                    # 1. 生成直线距离缓冲区
                    straight_buffer = center_point.buffer(distance_meters)
                    
                    # 2. 用路网蒙版裁剪缓冲区(核心: 路网限制)
                    roads_in_buffer = roads_buffered.intersection(straight_buffer)
                    
                    if roads_in_buffer.is_empty:
                        # 如果没有路网，使用简化的缓冲区
                        service_area = straight_buffer
                        failed_count += 1
                    else:
                        # 3. 对路网部分进行缓冲，形成连续的服务区
                        service_area = roads_in_buffer.buffer(30)  # 扩大路网影响范围
                        
                        # 4. 简化几何形状(使用凸包)
                        try:
                            if service_area.geom_type == 'MultiPolygon':
                                # 如果是多部分，取最大的部分
                                if hasattr(service_area, 'geoms'):
                                    largest = max(service_area.geoms, key=lambda g: g.area)
                                    service_area = largest.convex_hull
                                else:
                                    service_area = service_area.convex_hull
                            else:
                                service_area = service_area.convex_hull
                        except:
                            pass  # 如果凸包失败，保持原状
                    
                    # 准备服务区信息
                    service_info = {
                        'community_id': idx,
                        'community_idx': idx,  # 用于合并
                        'geometry': service_area
                    }
                    
                    # 添加小区信息
                    for field in ['name', 'id', 'longitude', 'latitude', 'convenience_score', 'convenience_level']:
                        if field in community:
                            if field in ['longitude', 'latitude']:
                                # 保存原始坐标
                                service_info[f'orig_{field}'] = community[field]
                            else:
                                service_info[field] = community[field]
                    
                    service_areas_list.append(service_info)
                    
                except Exception as e:
                    if verbose:
                        print(f"\n  ! 小区{idx}服务区生成失败: {e}")
                    failed_count += 1
                    continue
            
            # 创建服务区GeoDataFrame
            if service_areas_list:
                service_gdf = gpd.GeoDataFrame(service_areas_list, crs=communities_gdf.crs)
                
                # 计算统计信息
                areas = service_gdf.geometry.area
                area_stats = {
                    '平均面积': areas.mean(),
                    '最大面积': areas.max(),
                    '最小面积': areas.min(),
                    '总面积': areas.sum()
                }
                
                end_time = time.time()
                processing_time = end_time - start_time
                
                print("\n" + "=" * 60)
                print("☑ 服务区生成完成")
                print("-" * 60)
                print(f"成功生成: {len(service_gdf)}个服务区")
                print(f"生成失败: {failed_count}个")
                print(f"处理时间: {processing_time:.1f}秒")
                print(f"平均时间: {processing_time/len(communities_gdf):.2f}秒/个")
                print("\n面积统计:")
                for stat_name, stat_value in area_stats.items():
                    if stat_name == '总面积':
                        print(f"  {stat_name}: {stat_value:,.0f}平方米({stat_value/1e6:.1f}平方公里)")
                    else:
                        print(f"  {stat_name}: {stat_value:,.0f}平方米")
                        
                return service_gdf
            else:
                print("  X 未能生成任何服务区")
                return gpd.GeoDataFrame()
                
        except Exception as e:
            print(f"  X 服务区生成失败: {e}")
            import traceback
            traceback.print_exc()
            return gpd.GeoDataFrame()
            
    def count_poi_in_service_areas(self, service_areas_gdf, poi_gdf, verbose=True):
        """
        统计服务区内的POI数量
        修改：直接使用CSV中的category和sub_category字段进行分类，删除映射字典逻辑
        """
        print("\n" + "=" * 60)
        print("步骤3: 统计服务区内POI（基于CSV分类字段）")
        print("-" * 60)
        
        if service_areas_gdf.empty:
            print("  ! 服务区数据为空")
            return service_areas_gdf
            
        if poi_gdf.empty:
            print("  ! POI数据为空")
            return service_areas_gdf
            
        try:
            # 确保坐标系一致
            if service_areas_gdf.crs != poi_gdf.crs:
                if verbose:
                    print("统一POI数据坐标系...")
                poi_gdf = poi_gdf.to_crs(service_areas_gdf.crs)
            
            # 执行空间连接
            if verbose:
                print("执行空间连接(点在面内分析)...")
            # 使用空间连接找出每个POI点所在的服务区
            joined = gpd.sjoin(poi_gdf, service_areas_gdf, 
                             how="inner", predicate="within")
            
            if len(joined) == 0:
                print("  没有POI落在任何服务区内")
                # 添加空的POI计数列
                result_gdf = service_areas_gdf.copy()
                result_gdf['total_poi'] = 0
                return result_gdf
                
            if verbose:
                print(f"  空间连接成功: {len(joined)}条连接记录")
            
            # 对同一服务区内的相同POI去重
            if verbose:
                print("对重复POI进行去重...")
            
            # 确定去重的键
            if 'id' in joined.columns:
                # 使用POI的id去重
                joined_unique = joined.drop_duplicates(subset=['community_idx', 'id'])
                dup_key = 'id'
            elif 'name' in joined.columns and 'category' in joined.columns and 'sub_category' in joined.columns:
                # 使用名称、category和sub_category组合去重
                joined['poi_key'] = joined['name'] + '|' + joined['category'] + '|' + joined['sub_category'].fillna('')
                joined_unique = joined.drop_duplicates(
                    subset=['community_idx', 'poi_key']
                )
                dup_key = 'name+category+sub_category'
            elif 'name' in joined.columns and 'category' in joined.columns:
                # 使用名称和category组合去重
                joined['poi_key'] = joined['name'] + '|' + joined['category']
                joined_unique = joined.drop_duplicates(
                    subset=['community_idx', 'poi_key']
                )
                dup_key = 'name+category'
            elif 'name' in joined.columns:
                # 仅使用名称去重
                joined_unique = joined.drop_duplicates(
                    subset=['community_idx', 'name']
                )
                dup_key = 'name'
            else:
                # 无法去重，使用所有记录
                joined_unique = joined.copy()
                dup_key = '无'
            
            if verbose:
                print(f"  去重依据: {dup_key}")
                print(f"  去重前记录数: {len(joined)}")
                print(f"  去重后记录数: {len(joined_unique)}")
                print(f"  去除重复: {len(joined)-len(joined_unique)}条")
            
            # 按服务区和POI分类统计数量
            if verbose:
                print("按CSV分类字段统计POI数量...")
            
            # 创建POI分类标签 - 直接使用CSV中的category和sub_category字段
            if 'category' in joined_unique.columns:
                # 核心修改：创建分类标签，格式为"category_sub_category"或仅"category"
                def create_poi_label(row):
                    category = str(row['category']).strip()
                    # 检查sub_category是否存在且有效
                    if 'sub_category' in row and pd.notna(row['sub_category']):
                        sub_category = str(row['sub_category']).strip()
                        if sub_category and sub_category.lower() not in ['', 'nan', 'none']:
                            return f"{category}_{sub_category}"
                    return category
                
                joined_unique['poi_label'] = joined_unique.apply(create_poi_label, axis=1)
                
                # 按服务区和POI标签统计
                counts = joined_unique.groupby(['community_idx', 'poi_label']).size()
                counts = counts.reset_index(name='count')
                
                # 转换为宽表格式
                counts_wide = counts.pivot_table(
                    index='community_idx',
                    columns='poi_label',
                    values='count',
                    fill_value=0
                ).reset_index()
                
                # 重命名列，加上前缀
                counts_wide.columns = ['community_idx'] + \
                    [f'count_{col}' for col in counts_wide.columns if col != 'community_idx']
                
                # 合并统计结果回服务区数据
                result_gdf = service_areas_gdf.merge(
                    counts_wide,
                    left_on='community_idx',
                    right_on='community_idx',
                    how='left'
                )
                
                # 填充NaN为0
                count_cols = [col for col in result_gdf.columns if col.startswith('count_')]
                if count_cols:
                    result_gdf[count_cols] = result_gdf[count_cols].fillna(0)
                    result_gdf['total_poi'] = result_gdf[count_cols].sum(axis=1)
                
                # 计算POI密度(个/平方公里)
                result_gdf['area_km2'] = result_gdf.geometry.area / 1e6
                result_gdf['poi_density'] = result_gdf['total_poi'] / result_gdf['area_km2']
                result_gdf['poi_density'] = result_gdf['poi_density'].fillna(0)
            else:
                # 如果没有category字段，只统计总数
                print("  ! POI数据缺少category字段，仅统计总数")
                total_counts = joined_unique.groupby('community_idx').size()
                total_counts = total_counts.reset_index(name='total_poi')
                
                result_gdf = service_areas_gdf.merge(
                    total_counts,
                    left_on='community_idx',
                    right_on='community_idx',
                    how='left'
                )
                result_gdf['total_poi'] = result_gdf['total_poi'].fillna(0)
                result_gdf['area_km2'] = result_gdf.geometry.area / 1e6
                result_gdf['poi_density'] = result_gdf['total_poi'] / result_gdf['area_km2']
                result_gdf['poi_density'] = result_gdf['poi_density'].fillna(0)
            
            # 统计摘要
            if verbose:
                print("\n☑ POI统计摘要（基于CSV分类字段）:")
                print(f"  总POI数: {int(result_gdf['total_poi'].sum())}")
                print(f"  平均每个服务区POI数: {result_gdf['total_poi'].mean():.1f}")
                if 'poi_density' in result_gdf.columns:
                    print(f"  平均POI密度: {result_gdf['poi_density'].mean():.1f}个/平方公里")
                
                # 显示各类POI的总数
                count_cols = [col for col in result_gdf.columns if col.startswith('count_')]
                if count_cols:
                    print(f"\n  各类POI数量:")
                    for col in sorted(count_cols):
                        total = int(result_gdf[col].sum())
                        if total > 0:
                            poi_type = col.replace('count_', '')
                            print(f"  {poi_type}: {total}")
            
            return result_gdf
            
        except Exception as e:
            print(f"  X POI统计失败: {e}")
            import traceback
            traceback.print_exc()
            return service_areas_gdf

# 创建空间分析器实例
spatial_analyzer = SpatialAnalyzer(config)
print("空间分析器初始化完成")

空间分析器初始化完成


In [6]:
# 6. 结果可视化器 - 完整修复版
class ResultVisualizer:
    """结果可视化器"""
    
    def __init__(self, config):
        self.config = config
        
        # 中文字体配置
        import matplotlib
        matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'FangSong', 'KaiTi']
        matplotlib.rcParams['axes.unicode_minus'] = False
        
        # 创建输出目录
        self.figures_dir = config.figures_dir
        self.figures_dir.mkdir(exist_ok=True)
    
    def plot_service_areas_distribution(self, boundary_gdf, roads_gdf, 
                                        communities_gdf, service_areas_gdf,
                                        title="基于路网服务区的便利度空间分布",
                                        figsize=(16, 12),
                                        save=True):
        """
        绘制服务区分布图 - 修复版，解决左上角重叠问题
        """
        print("绘制服务区分布图...")
        fig, ax = plt.subplots(1, 1, figsize=figsize)
        
        # 1. 绘制底图要素
        boundary_gdf.plot(ax=ax, color='lightgray', edgecolor='black', 
                         alpha=0.3, linewidth=1, label='行政区边界')
        roads_gdf.plot(ax=ax, linewidth=0.8, color='darkgray', 
                      alpha=0.6, label='道路网络')
        
        # 2. 绘制服务区（按便利度着色）
        if not service_areas_gdf.empty and 'convenience_score' in service_areas_gdf.columns:
            cmap = LinearSegmentedColormap.from_list(
                'convenience_cmap',
                ['#d7191c', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641']
            )
            
            scores = service_areas_gdf['convenience_score']
            norm = Normalize(vmin=scores.min(), vmax=scores.max())
            
            for idx, row in service_areas_gdf.iterrows():
                if pd.notnull(row.geometry):
                    color = cmap(norm(row['convenience_score']))
                    gpd.GeoSeries([row.geometry]).plot(
                        ax=ax, color=color, alpha=0.7,
                        edgecolor='white', linewidth=0.5
                    )
            
            # 添加颜色条
            sm = ScalarMappable(cmap=cmap, norm=norm)
            sm.set_array([])
            cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
            cbar.set_label('便利度得分', fontsize=12)
        
        # 3. 绘制小区点
        communities_gdf.plot(ax=ax, markersize=40, color='black', marker='o',
                           alpha=0.8, label='居民小区', edgecolor='white', linewidth=1.5)
        
        # 4. 添加在线底图
        try:
            ctx.add_basemap(ax, crs=boundary_gdf.crs,
                          source=ctx.providers.CartoDB.Positron)
        except Exception as e:
            print(f"无法添加在线底图: {e}")
        
        # 5. 添加指北针和比例尺 - 调整到右上角
        if not boundary_gdf.empty:
            bounds = boundary_gdf.total_bounds
            try:
                # 调整指北针到右上角，避免与图例重叠
                arrow_x, arrow_y = 0.95, 0.95
                arrow = mpatches.FancyArrowPatch(
                    (arrow_x, arrow_y - 0.02),
                    (arrow_x, arrow_y + 0.02),
                    arrowstyle='->,head_width=0.5,head_length=0.7',
                    linewidth=2.0,
                    edgecolor='black',
                    facecolor='black',
                    transform=ax.transAxes,
                    zorder=1000
                )
                ax.add_patch(arrow)
                ax.text(arrow_x, arrow_y + 0.025, 'N',
                       transform=ax.transAxes,
                       ha='center', va='bottom',
                       fontsize=14, fontweight='bold',
                       bbox=dict(boxstyle="round,pad=0.2",
                                facecolor="white",
                                edgecolor="black",
                                alpha=0.8),
                       zorder=1000)
                
                # 添加比例尺到指北针下方
                scale_x, scale_y = 0.95, 0.88
                x_range = bounds[2] - bounds[0]
                approx_km_per_degree = 111.0
                scale_km = 2.0
                scale_frac = (scale_km / approx_km_per_degree) / x_range
                scale_frac = min(scale_frac, 0.1)
                
                # 绘制比例尺背景
                scale_bg = Rectangle(
                    (scale_x - scale_frac/2 - 0.01, scale_y - 0.015),
                    scale_frac + 0.02, 0.03,
                    transform=ax.transAxes,
                    facecolor='white',
                    edgecolor='black',
                    alpha=0.8,
                    linewidth=0.5,
                    zorder=999
                )
                ax.add_patch(scale_bg)
                
                # 绘制比例尺
                for i in range(5):
                    start = scale_x - scale_frac/2 + (i * scale_frac / 5)
                    end = start + scale_frac / 10
                    color = 'black' if i % 2 == 0 else 'white'
                    rect = Rectangle(
                        (start, scale_y - 0.01), scale_frac/10, 0.02,
                        transform=ax.transAxes,
                        facecolor=color,
                        edgecolor='black',
                        linewidth=0.5,
                        zorder=1000
                    )
                    ax.add_patch(rect)
                
                ax.text(scale_x, scale_y - 0.025, f'{scale_km} km',
                       transform=ax.transAxes,
                       ha='center', va='top',
                       fontsize=10, fontweight='bold',
                       bbox=dict(boxstyle="round,pad=0.1",
                                facecolor="white",
                                edgecolor="black",
                                alpha=0.8),
                       zorder=1000)
            except Exception as e:
                print(f"添加指北针/比例尺时出错: {e}")
        
        # 6. 设置图表属性
        ax.set_title(title, fontsize=18, fontweight='bold', pad=20)
        
        # 7. 优化图例位置和样式 - 放置到左下角，避免重叠
        handles, labels = ax.get_legend_handles_labels()
        
        if 'convenience_score' in service_areas_gdf.columns:
            from matplotlib.patches import Patch
            service_area_patch = Patch(facecolor='#a6d96a', alpha=0.7,
                                      edgecolor='white', linewidth=0.5,
                                      label='服务区（按便利度着色）')
            handles.append(service_area_patch)
            labels.append('服务区（按便利度着色）')
        
        # 移除重复图例项
        unique_labels = []
        unique_handles = []
        for handle, label in zip(handles, labels):
            if label not in unique_labels:
                unique_labels.append(label)
                unique_handles.append(handle)
        
        # 将图例放置在左下角
        if unique_handles:
            ax.legend(unique_handles, unique_labels,
                     loc='lower left', fontsize=10,
                     framealpha=0.9, edgecolor='black')
        
        ax.set_axis_off()
        plt.tight_layout()
        
        # 8. 保存图片
        if save:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"service_areas_distribution_{timestamp}.png"
            filepath = self.figures_dir / filename
            plt.savefig(filepath, dpi=300, bbox_inches='tight',
                       facecolor='white', edgecolor='none')
            plt.close()
            print(f"√ 图片已保存: {filepath}")
            return str(filepath)
        else:
            plt.show()
            return None
    
    def plot_convenience_distribution(self, result_df, save=True):
        """
        绘制便利度分布图表 - 拆分为两个独立的图表，避免重叠
        修改：删除了饼图部分
        """
        print("绘制便利度分布图表...")
        saved_files = []
        
        if 'convenience_score' not in result_df.columns:
            print("! 数据中不包含便利度得分")
            return saved_files
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # 1. 综合便利度得分分布直方图 - 独立图表
        fig1, ax1 = plt.subplots(1, 1, figsize=(10, 6))
        n_bins = min(30, len(result_df) // 5)
        n_bins = max(n_bins, 10)
        
        ax1.hist(result_df['convenience_score'], bins=n_bins,
                edgecolor='black', alpha=0.7, color='steelblue')
        ax1.set_xlabel('综合便利度得分', fontsize=12)
        ax1.set_ylabel('小区数量', fontsize=12)
        ax1.set_title('(A) 分数分布', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3, linestyle='--')
        
        # 添加统计信息 - 调整位置避免重叠
        mean_score = result_df['convenience_score'].mean()
        median_score = result_df['convenience_score'].median()
        std_score = result_df['convenience_score'].std()
        
        ax1.axvline(mean_score, color='red', linestyle='--', linewidth=2,
                   label=f'平均值: {mean_score:.3f}')
        ax1.axvline(median_score, color='green', linestyle='--', linewidth=2,
                   label=f'中位数: {median_score:.3f}')
        
        # 添加统计信息文本，调整位置避免重叠
        stats_text = f'N = {len(result_df)}\nμ = {mean_score:.3f}\nM = {median_score:.3f}\nσ = {std_score:.3f}'
        ax1.text(0.05, 0.95, stats_text,
                transform=ax1.transAxes, fontsize=10,
                verticalalignment='top', horizontalalignment='left',
                bbox=dict(boxstyle="round,pad=0.3",
                         facecolor="white",
                         edgecolor="black",
                         alpha=0.8))
        ax1.legend(loc='upper right', fontsize=10)
        plt.tight_layout()
        
        if save:
            filename1 = f"score_distribution_{timestamp}.png"
            filepath1 = self.figures_dir / filename1
            plt.savefig(filepath1, dpi=300, bbox_inches='tight')
            plt.close()
            saved_files.append(str(filepath1))
            print(f"√ 得分分布图已保存: {filepath1}")
        
        # 2. 供给-多样性维度散点图 - 独立图表
        if all(col in result_df.columns for col in ['supply_dimension', 'diversity_dimension']):
            fig3, ax3 = plt.subplots(1, 1, figsize=(12, 8))
            
            # 根据便利度等级着色
            if 'convenience_level' in result_df.columns:
                level_colors = {
                    '低便利度': '#d7191c',
                    '较低便利度': '#fdae61',
                    '中等便利度': '#ffffbf',
                    '较高便利度': '#a6d96a',
                    '高便利度': '#1a9641'
                }
                colors = result_df['convenience_level'].map(level_colors)
                
                # 分别绘制每个等级的点，以便添加图例
                for level, color in level_colors.items():
                    if level in result_df['convenience_level'].unique():
                        mask = result_df['convenience_level'] == level
                        ax3.scatter(result_df.loc[mask, 'supply_dimension'],
                                  result_df.loc[mask, 'diversity_dimension'],
                                  c=[color], s=50, alpha=0.7,
                                  edgecolors='black', linewidth=0.5,
                                  label=level)
            else:
                # 使用便利度得分着色
                scatter = ax3.scatter(result_df['supply_dimension'],
                                    result_df['diversity_dimension'],
                                    c=result_df['convenience_score'],
                                    cmap='RdYlGn',
                                    s=50, alpha=0.7, edgecolors='black',
                                    linewidth=0.5)
                cbar = plt.colorbar(scatter, ax=ax3)
                cbar.set_label('综合便利度得分', fontsize=12)
            
            ax3.set_xlabel('供给维度得分', fontsize=12)
            ax3.set_ylabel('多样性维度得分', fontsize=12)
            ax3.set_title('(B) 供给-多样性维度散点图', fontsize=14, fontweight='bold')
            ax3.grid(True, alpha=0.3, linestyle='--')
            
            # 添加回归线
            try:
                from scipy.stats import linregress
                slope, intercept, r_value, p_value, std_err = linregress(
                    result_df['supply_dimension'],
                    result_df['diversity_dimension']
                )
                x_line = np.linspace(result_df['supply_dimension'].min(),
                                   result_df['supply_dimension'].max(), 100)
                y_line = slope * x_line + intercept
                ax3.plot(x_line, y_line, 'r--', linewidth=1.5,
                       label=f'回归线 (R²={r_value**2:.3f})')
            except:
                pass
            
            # 添加图例 - 将图例放在图表外面（右侧）
            if 'convenience_level' in result_df.columns:
                # 创建图例
                legend = ax3.legend(title="便利度等级", loc='center left',
                                  bbox_to_anchor=(1.02, 0.5), fontsize=10)
                
                # 在便利度等级说明下方添加回归统计信息
                try:
                    from scipy.stats import linregress
                    slope, intercept, r_value, p_value, std_err = linregress(
                        result_df['supply_dimension'],
                        result_df['diversity_dimension']
                    )
                    
                    # 将p值显示为标准的科学计数法格式：2.139×10^{-8}
                    if p_value < 0.0001:
                        p_str = f"p = {p_value:.3e}".replace("e-0", "×10^{-").replace("e-", "×10^{-") + "}"
                    else:
                        p_str = f"p = {p_value:.4f}"
                    
                    reg_text = f'回归方程: y = {slope:.3f}x + {intercept:.3f}\nR² = {r_value**2:.3f}\n{p_str}'
                    
                    # 将回归统计信息放在图例下方
                    ax3.text(1.02, 0.35, reg_text,
                           transform=ax3.transAxes, fontsize=10,
                           verticalalignment='top', horizontalalignment='left',
                           bbox=dict(boxstyle="round,pad=0.3",
                                    facecolor="white",
                                    edgecolor="black",
                                    alpha=0.8))
                except:
                    pass
                
                # 调整子图位置，为右侧图例和回归统计信息留出空间
                plt.subplots_adjust(right=0.72)
            else:
                ax3.legend(loc='upper left', fontsize=10)
            
            # 设置坐标轴范围
            ax3.set_xlim([result_df['supply_dimension'].min() - 0.05,
                         result_df['supply_dimension'].max() + 0.05])
            ax3.set_ylim([result_df['diversity_dimension'].min() - 0.05,
                         result_df['diversity_dimension'].max() + 0.05])
            plt.tight_layout()
            
            if save:
                filename3 = f"supply_diversity_scatter_{timestamp}.png"
                filepath3 = self.figures_dir / filename3
                plt.savefig(filepath3, dpi=300, bbox_inches='tight')
                plt.close()
                saved_files.append(str(filepath3))
                print(f"√ 散点图已保存: {filepath3}")
        
        return saved_files
    
    def plot_top_bottom_communities(self, result_df, n=10, figsize=(14, 8), save=True):
        """
        绘制便利度最高和最低的小区 - 中文版
        """
        if 'convenience_score' not in result_df.columns or 'name' not in result_df.columns:
            print("! 数据中缺少必要列")
            return None
        
        # 获取便利度最高和最低的小区
        top_n = result_df.nlargest(n, 'convenience_score')
        bottom_n = result_df.nsmallest(n, 'convenience_score')
        
        # 设置颜色
        top_color = '#2ecc71'  # 绿色
        bottom_color = '#e74c3c'  # 红色
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize)
        
        # 便利度最高的N个小区
        y_pos1 = range(len(top_n))
        bars1 = ax1.barh(y_pos1, top_n['convenience_score'],
                        color=top_color, alpha=0.7, edgecolor='black')
        ax1.set_yticks(y_pos1)
        ax1.set_yticklabels(top_n['name'], fontsize=10)
        ax1.invert_yaxis()
        ax1.set_xlabel('便利度得分', fontsize=12)
        ax1.set_title(f'便利度最高的 {n} 个小区', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3, axis='x', linestyle='--')
        ax1.set_xlim([0, max(top_n['convenience_score'].max() * 1.1, 1.0)])
        
        # 在条形末端标注得分
        for i, (idx, row) in enumerate(top_n.iterrows()):
            score_text = f"{row['convenience_score']:.3f}"
            ax1.text(row['convenience_score'] + 0.01, i,
                    score_text, va='center', fontsize=9, fontweight='bold')
        
        # 便利度最低的N个小区
        y_pos2 = range(len(bottom_n))
        bars2 = ax2.barh(y_pos2, bottom_n['convenience_score'],
                        color=bottom_color, alpha=0.7, edgecolor='black')
        ax2.set_yticks(y_pos2)
        ax2.set_yticklabels(bottom_n['name'], fontsize=10)
        ax2.invert_yaxis()
        ax2.set_xlabel('便利度得分', fontsize=12)
        ax2.set_title(f'便利度最低的 {n} 个小区', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='x', linestyle='--')
        ax2.set_xlim([0, max(bottom_n['convenience_score'].max() * 1.1, 1.0)])
        
        # 在条形末端标注得分
        for i, (idx, row) in enumerate(bottom_n.iterrows()):
            score_text = f"{row['convenience_score']:.3f}"
            ax2.text(row['convenience_score'] + 0.01, i,
                    score_text, va='center', fontsize=9, fontweight='bold')
        
        plt.tight_layout()
        
        if save:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"top_bottom_communities_{timestamp}.png"
            filepath = self.figures_dir / filename
            plt.savefig(filepath, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"√ TOP小区图已保存: {filepath}")
            return str(filepath)
        else:
            plt.show()
            return None
    
    def plot_3d_dimensions(self, result_df, figsize=(16, 12), save=True):
        """
        绘制三维便利度维度空间分布图 - 修复Z轴标签显示问题
        将整个图形向右移动，为Z轴标签腾出空间
        """
        # 检查必需的维度列是否存在
        required_cols = ['supply_dimension', 'diversity_dimension', 'accessibility_dimension']
        missing_cols = [col for col in required_cols if col not in result_df.columns]
        
        if missing_cols:
            print(f"! 数据中缺少以下维度得分列: {missing_cols}")
            print(f"可用的列: {list(result_df.columns)}")
            
            # 尝试自动识别维度列
            dim_mapping = {}
            for col in result_df.columns:
                col_lower = str(col).lower()
                if any(word in col_lower for word in ['supply', '供给']):
                    dim_mapping['supply'] = col
                elif any(word in col_lower for word in ['diversity', '多样性']):
                    dim_mapping['diversity'] = col
                elif any(word in col_lower for word in ['accessibility', '可达性', 'access', '可达']):
                    dim_mapping['accessibility'] = col
                elif any(word in col_lower for word in ['proximity', '邻近性', '距离']):
                    dim_mapping['accessibility'] = col
            
            if len(dim_mapping) >= 3:
                supply_col = dim_mapping.get('supply')
                diversity_col = dim_mapping.get('diversity')
                accessibility_col = dim_mapping.get('accessibility')
                print(f"自动识别的维度列: 供给={supply_col}, 多样性={diversity_col}, 可达性={accessibility_col}")
            else:
                print("错误：需要至少三个维度列来创建三维图")
                return None
        else:
            supply_col = 'supply_dimension'
            diversity_col = 'diversity_dimension'
            accessibility_col = 'accessibility_dimension'
        
        try:
            from mpl_toolkits.mplot3d import Axes3D
            
            print("绘制三维便利度维度空间分布图...")
            print("注意：将图形向右移动，为Z轴标签腾出空间")
            
            # 创建图形，显著增加宽度
            fig = plt.figure(figsize=figsize)
            
            # 创建三维子图，但设置位置使其靠右
            ax = fig.add_axes([0.15, 0.1, 0.7, 0.8], projection='3d')  # 左、下、宽、高
            
            # 定义等级颜色
            level_colors = {
                '低便利度': '#d7191c',      # 深红色
                '较低便利度': '#fdae61',    # 橙色
                '中等便利度': '#ffffbf',    # 米黄色
                '较高便利度': '#a6d96a',    # 浅绿色
                '高便利度': '#1a9641'       # 深绿色
            }
            
            # 准备颜色和标签
            if 'convenience_level' in result_df.columns:
                colors = result_df['convenience_level'].map(level_colors)
                legend_title = '便利度等级'
                
                from matplotlib.patches import Patch
                legend_elements = []
                for level, color in level_colors.items():
                    if level in result_df['convenience_level'].unique():
                        legend_elements.append(Patch(facecolor=color, label=level, edgecolor='black'))
                
                print(f"使用便利度等级着色，等级分布: {result_df['convenience_level'].value_counts().to_dict()}")
            elif 'convenience_score' in result_df.columns:
                from matplotlib.cm import ScalarMappable
                from matplotlib.colors import Normalize
                scores = result_df['convenience_score']
                norm = Normalize(vmin=scores.min(), vmax=scores.max())
                cmap = plt.cm.RdYlGn
                colors = cmap(norm(scores))
                legend_title = '便利度得分'
                print(f"使用便利度得分连续着色，得分范围: {scores.min():.3f} - {scores.max():.3f}")
            else:
                colors = '#3498db'
                legend_title = None
            
            # 绘制散点图 - 优化视觉参数
            scatter = ax.scatter(
                result_df[diversity_col],    # X轴：多样性维度
                result_df[supply_col],       # Y轴：供给维度
                result_df[accessibility_col], # Z轴：可达性维度
                c=colors,
                s=25,           # 减小点的大小
                alpha=0.6,      # 增加透明度
                edgecolors='black',
                linewidth=0.3,  # 减小边线宽度
                depthshade=True
            )
            
            # 设置坐标轴标签
            ax.set_xlabel('多样性维度', fontsize=12, labelpad=10)
            ax.set_ylabel('供给维度', fontsize=12, labelpad=10)
            ax.set_zlabel('可达性维度', fontsize=12, labelpad=15)  # 增加Z轴标签边距
            
            ax.set_title('三维便利度维度空间分布', fontsize=14, fontweight='bold', pad=20)
            
            # 优化视角
            ax.view_init(elev=20, azim=45)
            
            # 优化网格
            ax.grid(True, alpha=0.2, linestyle='--')
            
            # 调整刻度标签
            ax.tick_params(axis='x', labelsize=9, pad=5)
            ax.tick_params(axis='y', labelsize=9, pad=5)
            ax.tick_params(axis='z', labelsize=9, pad=8)  # Z轴标签增加内边距
            
            # 设置坐标轴范围，留出适当边距
            x_min, x_max = result_df[diversity_col].min(), result_df[diversity_col].max()
            y_min, y_max = result_df[supply_col].min(), result_df[supply_col].max()
            z_min, z_max = result_df[accessibility_col].min(), result_df[accessibility_col].max()
            
            x_range = x_max - x_min
            y_range = y_max - y_min
            z_range = z_max - z_min
            
            margin_x = x_range * 0.1 if x_range > 0 else 0.1
            margin_y = y_range * 0.1 if y_range > 0 else 0.1
            margin_z = z_range * 0.1 if z_range > 0 else 0.1
            
            ax.set_xlim([x_min - margin_x, x_max + margin_x])
            ax.set_ylim([y_min - margin_y, y_max + margin_y])
            ax.set_zlim([z_min - margin_z, z_max + margin_z])
            
            # 添加图例 - 调整位置
            if 'convenience_level' in result_df.columns and 'legend_elements' in locals():
                ax.legend(
                    handles=legend_elements,
                    title=legend_title,
                    loc='upper left',
                    bbox_to_anchor=(1.05, 1.0),  # 放在图形右侧
                    fontsize=10,
                    framealpha=0.9,
                    edgecolor='black'
                )
            elif 'convenience_score' in result_df.columns and 'cmap' in locals():
                sm = ScalarMappable(cmap=cmap, norm=norm)
                sm.set_array([])
                cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.12)
                cbar.set_label(legend_title, fontsize=12)
            
            # 输出维度统计信息
            print(f"\n三维图维度统计:")
            print(f"  X轴 - 多样性维度: {x_min:.3f} 到 {x_max:.3f}")
            print(f"  Y轴 - 供给维度: {y_min:.3f} 到 {y_max:.3f}")
            print(f"  Z轴 - 可达性维度: {z_min:.3f} 到 {z_max:.3f}")
            
            if save:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"3d_dimensions_distribution_{timestamp}.png"
                filepath = self.figures_dir / filename
                
                # 保存时增加右侧内边距
                plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white', 
                          pad_inches=0.5)
                plt.close()
                print(f"√ 三维维度分布图已保存: {filepath}")
                return str(filepath)
            else:
                plt.show()
                return None
        
        except ImportError as e:
            print(f"无法导入3D绘图库: {e}")
            return None
        except Exception as e:
            print(f"X 绘制三维图失败: {e}")
            return None
    
    def generate_all_visualizations(self, boundary_gdf, roads_gdf, communities_gdf,
                                   service_areas_gdf, result_df):
        """
        生成所有可视化图表
        修改：删除了热力图生成
        """
        print("\n" + "=" * 60)
        print("生成全套可视化图表")
        print("=" * 60)
        
        visualizations = {}
        
        try:
            # 1. 服务区分布图
            print("1. 生成服务区分布图...")
            map1 = self.plot_service_areas_distribution(
                boundary_gdf, roads_gdf, communities_gdf, service_areas_gdf)
            if map1:
                visualizations['service_areas_map'] = map1
                print("√ 服务区分布图生成完成")
            
            # 2. 便利度分布图表（拆分为两个独立图表）
            print("2. 生成便利度分布图表...")
            dist_charts = self.plot_convenience_distribution(result_df)
            if dist_charts:
                visualizations['distribution_charts'] = dist_charts
                print(f"√ 生成 {len(dist_charts)} 个分布图表")
            
            # 3. 三维维度分布图
            print("3. 生成三维维度分布图...")
            map_3d = self.plot_3d_dimensions(result_df)
            if map_3d:
                visualizations['3d_dimensions_map'] = map_3d
                print("√ 三维维度分布图生成完成")
            
            # 4. TOP/BOTTOM小区图
            print("4. 生成TOP/BOTTOM小区图...")
            top_bottom_map = self.plot_top_bottom_communities(result_df, n=10)
            if top_bottom_map:
                visualizations['top_bottom_communities'] = top_bottom_map
                print("√ TOP/BOTTOM小区图生成完成")
            
            print("\n☑ 所有可视化图表生成完成")
            print(f"共生成 {len(visualizations)} 类图表")
            return visualizations
            
        except Exception as e:
            print(f"X 可视化生成失败: {e}")
            import traceback
            traceback.print_exc()
            return visualizations

# 创建可视化器实例
visualizer = ResultVisualizer(config)
print("☑ 结果可视化器初始化完成")

☑ 结果可视化器初始化完成


In [7]:
# 7. CRITIC法便利度计算模块 - 修复版，确保使用真正的自然断点法
class CRITICConvenienceCalculator:
    """
    CRITIC法便利度计算器 - 基于CRITIC客观赋权法的多维综合评价
    修复：确保真正使用自然断点法，并提供备用方案
    """
    
    def __init__(self, config):
        self.config = config
        self.scaler = MinMaxScaler()
    
    def calculate_critic_weights(self, poi_counts_df, verbose=True):
        """
        使用CRITIC法计算指标权重
        
        参数:
            poi_counts_df: 包含POI计数数据的DataFrame
            verbose: 是否显示详细过程
            
        返回:
            dict: 各指标权重字典
        """
        if verbose:
            print("\n" + "=" * 60)
            print("步骤: 使用CRITIC法计算指标权重")
            print("=" * 60)
        
        # 获取POI计数列
        count_cols = [col for col in poi_counts_df.columns
                     if col.startswith('count_') and col != 'count_community_idx']
        
        if not count_cols:
            if verbose:
                print("未找到POI计数列，无法计算权重")
            return {}
        
        # 提取数据
        data = poi_counts_df[count_cols].values.astype(float)
        
        # 1. 数据标准化（Min-Max标准化）
        data_norm = (data - data.min(axis=0)) / (data.max(axis=0) - data.min(axis=0) + 1e-10)
        
        # 2. 计算标准差（变异性）
        std_values = np.std(data_norm, axis=0, ddof=1)
        
        # 3. 计算相关系数矩阵
        corr_matrix = np.corrcoef(data_norm, rowvar=False)
        
        # 4. 计算冲突性
        conflict = np.sum(1 - np.abs(corr_matrix), axis=0)
        
        # 5. 计算信息量
        information = std_values * conflict
        
        # 6. 计算权重
        weights = information / np.sum(information)
        
        # 创建权重字典
        simple_cols = [col.replace('count_', '') for col in count_cols]
        weight_dict = {simple_cols[i]: weights[i] for i in range(len(simple_cols))}
        
        if verbose:
            print("CRITIC法计算结果:")
            print("-" * 40)
            print(f"{'指标':<15} {'标准差':<10} {'冲突性':<10} {'信息量':<10} {'权重':<10}")
            print("-" * 40)
            for i, col in enumerate(simple_cols):
                print(f"{col:<15} {std_values[i]:<10.4f} {conflict[i]:<10.4f} "
                      f"{information[i]:<10.4f} {weights[i]:<10.4f}")
            
            # 按权重排序显示
            print("\n按权重排序:")
            sorted_weights = sorted(weight_dict.items(), key=lambda x: x[1], reverse=True)
            for poi_type, weight in sorted_weights:
                print(f"  {poi_type}: {weight:.4f}")
            
            # 保存权重结果
            weights_df = pd.DataFrame({
                'POI类型': simple_cols,
                '标准差': std_values,
                '冲突性': conflict,
                '信息量': information,
                'CRITIC权重': weights
            }).sort_values('CRITIC权重', ascending=False)
            
            weights_path = self.config.reports_dir / "CRITIC权重计算结果.csv"
            weights_df.to_csv(weights_path, index=False, encoding='utf-8-sig')
            if verbose:
                print(f"\n√ 权重结果已保存: {weights_path}")
            
            print("=" * 60)
        
        return weight_dict
    
    def _classify_with_natural_breaks(self, scores, n_classes=5):
        """
        使用自然断点法进行分类
        如果自然断点法失败，提供多种备选方案
        """
        # 先尝试使用jenkspy库的自然断点法
        try:
            import jenkspy
            
            # 使用Jenks自然断点法
            breaks = jenkspy.jenks_breaks(scores, n_classes=n_classes)
            
            # 检查断点数量是否正确
            if len(breaks) == n_classes + 1:
                return breaks
            else:
                print(f"警告: jenkspy返回的断点数量异常: {len(breaks)}个，期望{n_classes+1}个")
                raise ValueError("断点数量异常")
                
        except ImportError:
            print("警告: jenkspy库未安装，尝试使用备用方案")
            print("请安装jenkspy库: pip install jenkspy")
        except Exception as e:
            print(f"jenkspy自然断点法失败: {e}")
        
        # 备选方案1: 使用Fisher-Jenks算法的手动实现
        print("尝试使用Fisher-Jenks算法的简化实现...")
        
        def simple_fisher_jenks(data, n_classes):
            """简化的Fisher-Jenks算法"""
            # 对数据进行排序
            sorted_data = np.sort(data)
            n = len(sorted_data)
            
            # 初始划分：等分
            initial_breaks = np.linspace(sorted_data[0], sorted_data[-1], n_classes + 1)
            
            # 使用K-means的变体进行优化
            from scipy.cluster.vq import kmeans
            centroids, _ = kmeans(sorted_data.reshape(-1, 1), n_classes)
            centroids = np.sort(centroids.flatten())
            
            # 基于中心点确定边界
            boundaries = []
            for i in range(len(centroids) - 1):
                boundary = (centroids[i] + centroids[i+1]) / 2
                boundaries.append(boundary)
            
            # 添加最小值和最大值
            breaks = [sorted_data[0]] + boundaries + [sorted_data[-1]]
            return breaks
        
        try:
            breaks = simple_fisher_jenks(scores, n_classes)
            print(f"简化Fisher-Jenks算法成功，断点: {[f'{x:.3f}' for x in breaks]}")
            return breaks
        except Exception as e:
            print(f"简化Fisher-Jenks算法失败: {e}")
        
        # 备选方案2: 使用头尾法（Head/tail Breaks）适用于重尾分布
        def head_tail_breaks(data, n_classes):
            """头尾断点法，适用于重尾分布数据"""
            sorted_data = np.sort(data)
            breaks = [sorted_data[0]]
            
            current_data = sorted_data
            for _ in range(n_classes - 1):
                if len(current_data) <= 1:
                    break
                mean_val = np.mean(current_data)
                breaks.append(mean_val)
                # 保留大于均值的数据（头部）
                current_data = current_data[current_data > mean_val]
            
            breaks.append(sorted_data[-1])
            return breaks
        
        try:
            breaks = head_tail_breaks(scores, n_classes)
            print(f"头尾断点法成功，断点: {[f'{x:.3f}' for x in breaks]}")
            return breaks
        except Exception as e:
            print(f"头尾断点法失败: {e}")
        
        # 如果所有方法都失败，使用等分位数法但标记为特殊情况
        print("所有自然断点法都失败，使用分位数法作为最后手段")
        breaks = np.percentile(scores, np.linspace(0, 100, n_classes + 1))
        return breaks
    
    def calculate_convenience_score(self, poi_counts_df, weights, verbose=True):
        """
        计算综合便利度得分
        基于CRITIC法权重
        修复：确保真正使用自然断点法
        """
        if verbose:
            print("\n" + "=" * 60)
            print("步骤: 计算综合便利度得分（基于CRITIC法）")
            print("=" * 60)
        
        result_df = poi_counts_df.copy()
        
        # 1. 计算加权设施供给得分
        if verbose:
            print("1. 计算加权设施供给得分...")
        
        result_df['weighted_poi_score'] = 0
        for poi_type, weight in weights.items():
            col_name = f'count_{poi_type}'
            if col_name in result_df.columns:
                result_df['weighted_poi_score'] += result_df[col_name] * weight
        
        # 标准化加权得分
        if len(result_df) > 1 and result_df['weighted_poi_score'].std() > 0:
            result_df['weighted_poi_score_norm'] = self.scaler.fit_transform(
                result_df[['weighted_poi_score']]
            )
        else:
            result_df['weighted_poi_score_norm'] = 0.5
        
        # 2. 计算三个维度得分
        if verbose:
            print("2. 计算三个维度得分...")
        
        # 2.1 供给维度（加权数量40% + 密度60%）
        if 'weighted_poi_score_norm' in result_df.columns and 'poi_density' in result_df.columns:
            # 标准化POI密度
            if len(result_df) > 1 and result_df['poi_density'].std() > 0:
                result_df['poi_density_norm'] = self.scaler.fit_transform(
                    result_df[['poi_density']]
                )
            else:
                result_df['poi_density_norm'] = 0.5
            
            result_df['supply_dimension'] = (
                result_df['weighted_poi_score_norm'] * 0.4 +
                result_df['poi_density_norm'] * 0.6
            )
        elif 'weighted_poi_score_norm' in result_df.columns:
            result_df['supply_dimension'] = result_df['weighted_poi_score_norm']
        else:
            result_df['supply_dimension'] = 0.5
        
        # 2.2 多样性维度（香农多样性指数 + 辛普森均衡度）
        if verbose:
            print("计算多样性维度...")
        
        # 获取POI计数列
        count_cols = [col for col in result_df.columns 
                     if col.startswith('count_') and col != 'count_community_idx']
        
        if count_cols:
            def shannon_diversity(row):
                """计算香农多样性指数"""
                vals = row[count_cols].values.astype(float)
                total = vals.sum()
                if total == 0:
                    return 0
                proportions = vals[vals > 0] / total
                return -np.sum(proportions * np.log(proportions))
            
            def simpson_evenness(row):
                """计算辛普森均衡度指数"""
                vals = row[count_cols].values.astype(float)
                total = vals.sum()
                if total == 0:
                    return 0
                proportions = vals / total
                D = np.sum(proportions ** 2)  # 辛普森集中度
                return 1 - D  # 均衡度
            
            # 计算多样性指标
            result_df['diversity_shannon'] = result_df.apply(shannon_diversity, axis=1)
            result_df['diversity_simpson'] = result_df.apply(simpson_evenness, axis=1)
            
            # 标准化多样性指标
            if len(result_df) > 1:
                for col in ['diversity_shannon', 'diversity_simpson']:
                    if col in result_df.columns and result_df[col].std() > 0:
                        result_df[f'{col}_norm'] = self.scaler.fit_transform(result_df[[col]])
                    else:
                        result_df[f'{col}_norm'] = 0.5
            else:
                result_df['diversity_shannon_norm'] = 0.5
                result_df['diversity_simpson_norm'] = 0.5
            
            # 多样性维度（香农50% + 辛普森50%）
            result_df['diversity_dimension'] = (
                result_df.get('diversity_shannon_norm', 0.5) * 0.5 +
                result_df.get('diversity_simpson_norm', 0.5) * 0.5
            )
        else:
            result_df['diversity_dimension'] = 0.5
        
        # 2.3 可达性维度
        if verbose:
            print("计算可达性维度...")
        
        # 这里用POI密度作为可达性的代理指标
        if 'poi_density' in result_df.columns:
            if len(result_df) > 1 and result_df['poi_density'].std() > 0:
                result_df['accessibility_dimension'] = self.scaler.fit_transform(
                    result_df[['poi_density']]
                )
            else:
                result_df['accessibility_dimension'] = 0.5
        else:
            result_df['accessibility_dimension'] = 0.5
        
        # 3. 计算综合便利度得分
        if verbose:
            print("3. 计算综合便利度得分...")
        
        # 使用动态计算的维度权重
        dimension_weights = self.config.dimension_weights
        
        result_df['convenience_score'] = (
            result_df['supply_dimension'] * dimension_weights['supply'] +
            result_df['diversity_dimension'] * dimension_weights['diversity'] +
            result_df['accessibility_dimension'] * dimension_weights['access']
        )
        
        # 确保得分在[0, 1]范围内
        result_df['convenience_score'] = result_df['convenience_score'].clip(0, 1)
        
        # 4. 便利度分级 - 使用真正的自然断点法
        if verbose:
            print("4. 进行便利度分级（使用自然断点法）...")
        
        # 检查得分是否全相同
        unique_scores = result_df['convenience_score'].nunique()
        if unique_scores <= 1:
            # 所有得分相同，无法分箱
            print("警告: 所有小区便利度得分相同，无法进行分级")
            result_df['convenience_level'] = '中等便利度'
            classification_method = "单一值"
        elif len(result_df) < 5:
            # 如果数据太少，使用简单分级
            print("警告: 数据点少于5个，使用简单分级")
            def simple_level(score):
                if score >= 0.8:
                    return '高便利度'
                elif score >= 0.6:
                    return '较高便利度'
                elif score >= 0.4:
                    return '中等便利度'
                elif score >= 0.2:
                    return '较低便利度'
                else:
                    return '低便利度'
            
            result_df['convenience_level'] = result_df['convenience_score'].apply(simple_level)
            classification_method = "简单阈值"
        else:
            # 尝试使用自然断点法
            scores = result_df['convenience_score'].values
            
            try:
                # 获取自然断点
                breaks = self._classify_with_natural_breaks(scores, n_classes=5)
                
                # 确保断点数量正确且无重复
                if len(set(breaks)) < len(breaks):
                    print("警告: 断点有重复值，使用去重后的断点")
                    breaks = sorted(list(set(breaks)))
                
                # 确保有足够的断点
                if len(breaks) < 6:
                    # 如果断点不足，补充断点
                    print(f"警告: 断点数量不足 ({len(breaks)}个)，补充断点")
                    min_score, max_score = scores.min(), scores.max()
                    if len(breaks) == 2:
                        breaks = [min_score] + list(np.linspace(min_score, max_score, 4)[1:-1]) + [max_score]
                    else:
                        breaks = list(np.linspace(min_score, max_score, 6))
                
                # 使用自然断点进行分箱
                result_df['convenience_level'] = pd.cut(
                    result_df['convenience_score'],
                    bins=breaks,
                    labels=['低便利度', '较低便利度', '中等便利度', '较高便利度', '高便利度'],
                    include_lowest=True
                )
                
                classification_method = "自然断点法"
                
                if verbose:
                    print(f"自然断点法边界: {[f'{x:.3f}' for x in breaks]}")
                    intervals = [breaks[i+1]-breaks[i] for i in range(len(breaks)-1)]
                    print(f"边界间隔: {[f'{x:.3f}' for x in intervals]}")
                    
            except Exception as e:
                print(f"自然断点法失败，使用分位数分级: {e}")
                # 降级到分位数分箱
                result_df['convenience_level'] = pd.qcut(
                    result_df['convenience_score'],
                    q=5,
                    labels=['低便利度', '较低便利度', '中等便利度', '较高便利度', '高便利度'],
                    duplicates='drop'
                )
                classification_method = "分位数法（自然断点法失败）"
        
        # 5. 描述性统计
        if verbose:
            print(f"\n☑ 便利度得分统计 ({classification_method}):")
            print(f"样本总数: {len(result_df)}")
            print(f"最小值: {result_df['convenience_score'].min():.4f}")
            print(f"最大值: {result_df['convenience_score'].max():.4f}")
            print(f"平均值: {result_df['convenience_score'].mean():.4f}")
            print(f"中位数: {result_df['convenience_score'].median():.4f}")
            print(f"标准差: {result_df['convenience_score'].std():.4f}")
            
            # 检查数据分布均匀性
            sorted_scores = np.sort(result_df['convenience_score'].values)
            diffs = np.diff(sorted_scores)
            if len(diffs) > 0:
                uniformity = np.std(diffs) / (np.mean(diffs) + 1e-10)
                print(f"数据均匀性指标 (标准差/均值): {uniformity:.3f}")
                if uniformity < 0.1:
                    print("注意: 数据分布非常均匀，可能导致分级近似等分")
            
            print("\n☑ 便利度等级分布:")
            level_counts = result_df['convenience_level'].value_counts().sort_index()
            for level, count in level_counts.items():
                percentage = count / len(result_df) * 100
                print(f"  {level}: {count}个 ({percentage:.1f}%)")
            
            # 打印等级得分范围
            print("\n☑ 各等级得分范围:")
            for level in ['低便利度', '较低便利度', '中等便利度', '较高便利度', '高便利度']:
                if level in result_df['convenience_level'].values:
                    level_scores = result_df[result_df['convenience_level'] == level]['convenience_score']
                    if len(level_scores) > 0:
                        print(f"  {level}: {level_scores.min():.3f} - {level_scores.max():.3f}")
                    else:
                        print(f"  {level}: 无数据")
            
            print("\n各维度得分统计:")
            for dim in ['supply_dimension', 'diversity_dimension', 'accessibility_dimension']:
                if dim in result_df.columns:
                    mean_val = result_df[dim].mean()
                    std_val = result_df[dim].std()
                    print(f"  {dim}: {mean_val:.3f} (±{std_val:.3f})")
            
            # 验证分级方法
            print("\n分级方法验证:")
            print(f"  实际使用的方法: {classification_method}")
            
            # 检查是否等分
            level_counts_values = level_counts.values
            if len(level_counts_values) > 0:
                max_count = max(level_counts_values)
                min_count = min(level_counts_values)
                if max_count == min_count:
                    print("  警告: 所有等级数量相同，这可能是由于:")
                    print("  1. 数据分布非常均匀")
                    print("  2. 实际使用了等分方法")
                    print("  3. 样本数量正好是等级数的倍数")
                else:
                    imbalance_ratio = max_count / min_count
                    print(f"  等级间最大/最小数量比: {imbalance_ratio:.2f}")
            
            print("=" * 60)
        
        return result_df
    
    def get_top_bottom_communities(self, result_df, n=10):
        """
        获取便利度最高和最低的小区
        """
        if 'convenience_score' not in result_df.columns:
            return pd.DataFrame(), pd.DataFrame()
        
        # 获取便利度最高的N个小区
        top_df = result_df.nlargest(n, 'convenience_score')[['name', 'convenience_score', 'convenience_level']]
        
        # 获取便利度最低的N个小区
        bottom_df = result_df.nsmallest(n, 'convenience_score')[['name', 'convenience_score', 'convenience_level']]
        
        return top_df, bottom_df

# 创建CRITIC法便利度计算器实例
critic_calculator = CRITICConvenienceCalculator(config)
print("☑ CRITIC法便利度计算器初始化完成")

☑ CRITIC法便利度计算器初始化完成


In [8]:
# ============================================================================
# 7.2 权重敏感性分析模块 
# ============================================================================
class SensitivityAnalyzer:
    """
    权重敏感性分析器
    用于检验不同三维度权重组合对综合便利度得分的影响
    """
    def __init__(self, config):
        self.config = config
        
    def calculate_score_with_weights(self, result_df, weight_set, weight_name):
        """
        使用指定的权重组合重新计算综合便利度得分
        参数:
            result_df: 包含三个维度得分的DataFrame
            weight_set: 字典，格式如 {'supply': 0.4, 'diversity': 0.35, 'access': 0.25}
            weight_name: 该权重组合的名称，用于标识
        返回:
            Series: 新的综合便利度得分
        """
        # 确保所需维度列存在
        required_dims = ['supply_dimension', 'diversity_dimension', 'accessibility_dimension']
        if not all(dim in result_df.columns for dim in required_dims):
            # 尝试寻找可能的替代列名
            dim_mapping = {}
            for col in result_df.columns:
                col_lower = str(col).lower()
                if 'supply' in col_lower or '供给' in col_lower:
                    dim_mapping['supply'] = col
                elif 'diversity' in col_lower or '多样性' in col_lower:
                    dim_mapping['diversity'] = col
                elif 'access' in col_lower or '可达' in col_lower:
                    dim_mapping['access'] = col
            
            if len(dim_mapping) == 3:
                supply_col = dim_mapping['supply']
                diversity_col = dim_mapping['diversity']
                access_col = dim_mapping['access']
            else:
                raise ValueError(f"结果DataFrame中缺少必要的维度得分列。现有维度列: {dim_mapping}")
        else:
            supply_col, diversity_col, access_col = required_dims
        
        # 使用新权重计算综合得分
        new_score = (
            result_df[supply_col] * weight_set['supply'] +
            result_df[diversity_col] * weight_set['diversity'] +
            result_df[access_col] * weight_set['access']
        )
        # 确保得分在[0,1]范围内
        new_score = new_score.clip(0, 1)
        return new_score
    
    def perform_sensitivity_analysis(self, result_df, original_score_col='convenience_score', verbose=True):
        """
        执行完整的敏感性分析
        参数:
            result_df: 包含原始得分和三个维度得分的DataFrame
            original_score_col: 原始综合得分的列名
            verbose: 是否打印详细过程
        返回:
            DataFrame: 包含所有权重组合下得分的DataFrame
            dict: 包含与原始得分的皮尔逊相关系数字典
        """
        if verbose:
            print("\n" + "="*60)
            print("步骤6.4: 权重敏感性分析")
            print("-"*60)
        
        # 1. 定义多组权重组合
        weight_sets = {
            '原始预设': {'supply': 0.40, 'diversity': 0.35, 'access': 0.25},
            '等权重':   {'supply': 1/3,  'diversity': 1/3,  'access': 1/3},
            '供给优先': {'supply': 0.50, 'diversity': 0.30, 'access': 0.20},
            '均衡权重': {'supply': 0.35, 'diversity': 0.35, 'access': 0.30},
        }
        
        if verbose:
            print("定义的权重组合:")
            for name, weights in weight_sets.items():
                print(f"  - {name}: 供给={weights['supply']:.2f}, "
                      f"多样性={weights['diversity']:.2f}, 可达性={weights['access']:.2f}")
        
        # 2. 为每组权重重新计算综合得分
        sensitivity_results = result_df.copy()
        for name, weights in weight_sets.items():
            new_score = self.calculate_score_with_weights(sensitivity_results, weights, name)
            col_name = f'convenience_{name}'
            sensitivity_results[col_name] = new_score
            
            if verbose and name != '原始预设':
                print(f"\n√ '{name}' 权重组合计算完成:")
                print(f"  得分范围: {new_score.min():.3f} - {new_score.max():.3f}")
                print(f"  平均值: {new_score.mean():.3f}")
        
        # 3. 计算各新得分与原始得分的皮尔逊相关系数
        correlations = {}
        original_score = sensitivity_results[original_score_col]
        
        for name in weight_sets.keys():
            if name == '原始预设':
                corr, p_value = 1.0, 0.0
            else:
                new_score = sensitivity_results[f'convenience_{name}']
                # 计算皮尔逊相关系数和p值
                try:
                    from scipy.stats import pearsonr
                    corr, p_value = pearsonr(original_score, new_score)
                except ImportError:
                    # 备用方案
                    corr_matrix = np.corrcoef(original_score, new_score)
                    corr = corr_matrix[0, 1]
                    p_value = np.nan
        
            correlations[name] = {'相关系数': corr, 'p值': p_value}
        
        # 4. 打印分析结果
        if verbose:
            print("\n" + "-"*60)
            print("敏感性分析结果: 与原始得分的皮尔逊相关系数")
            print("-"*60)
            print(f"{'权重组合':<10} {'相关系数(r)':<12} {'p值':<12} {'显著性':<10}")
            print("-"*60)
            
            for name, corr_info in correlations.items():
                corr = corr_info['相关系数']
                p_val = corr_info['p值']
                sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "n.s."
                p_str = f"{p_val:.4f}" if not np.isnan(p_val) else "N/A"
                print(f"{name:<10} {corr:<12.4f} {p_str:<12} {sig:<10}")
            
            # 结论
            high_corr_count = sum(1 for info in correlations.values() if info['相关系数'] > 0.98)
            print(f"\n结论: 在 {len(correlations)-1} 组替代权重中，有 {high_corr_count} 组与原始得分的相关系数 > 0.98。")
            if high_corr_count == len(correlations) - 1:
                print("这表明综合评价结果对不同权重设定具有高度稳健性 (Robustness)。")
        
        # 5. 保存敏感性分析详细结果
        try:
            # 提取关键列
            score_cols = [original_score_col] + [f'convenience_{name}' for name in weight_sets.keys()]
            # 确保‘name’列存在，用于保存结果
            if 'name' in sensitivity_results.columns:
                save_cols = ['name'] + score_cols
            else:
                save_cols = score_cols
            
            sensitivity_path = self.config.reports_dir / "权重敏感性分析详细结果.csv"
            sensitivity_results[save_cols].to_csv(sensitivity_path, index=False, encoding='utf-8-sig')
            if verbose:
                print(f"\n√ 敏感性分析详细结果已保存: {sensitivity_path}")
        except Exception as e:
            if verbose:
                print(f"! 保存详细结果时发生轻微错误 (不影响主流程): {e}")

        return sensitivity_results, correlations

# 创建敏感性分析器实例
sensitivity_analyzer = SensitivityAnalyzer(config)
print("☑ 权重敏感性分析器初始化完成")

☑ 权重敏感性分析器初始化完成


In [9]:
# 8. 数据导出器
class DataExporter:
    """
    数据导出器 - 将分析结果保存为多种格式
    """
    
    def __init__(self, config):
        self.config = config
    
    def save_analysis_results(self, result_gdf, communities_gdf, prefix="analysis"):
        """
        保存分析结果为多种格式
        
        参数:
            result_gdf: 分析结果GeoDataFrame
            communities_gdf: 小区点位数据
            prefix: 文件名前缀
            
        返回:
            dict: 保存的文件路径
        """
        print("\n" + "=" * 60)
        print("步骤: 保存分析结果")
        print("-" * 60)
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        saved_files = {}
        
        try:
            # 1. 保存空间数据（Shapefile）
            print("保存空间数据...")
            shp_dir = self.config.output_dir / "shapefiles"
            shp_dir.mkdir(exist_ok=True)
            
            # 保存服务区数据
            service_area_shp = shp_dir / f"{prefix}_service_areas_{timestamp}.shp"
            result_gdf.to_file(service_area_shp, encoding='utf-8')
            saved_files['service_areas_shp'] = str(service_area_shp)
            print(f"√ 服务区Shapefile: {service_area_shp}")
            
            # 保存小区点位数据
            communities_shp = shp_dir / f"{prefix}_communities_{timestamp}.shp"
            communities_gdf.to_file(communities_shp, encoding='utf-8')
            saved_files['communities_shp'] = str(communities_shp)
            print(f"√ 小区点位Shapefile: {communities_shp}")
            
            # 2. 保存GeoJSON（用于Web地图）
            print("\n保存GeoJSON数据...")
            geojson_dir = self.config.output_dir / "geojson"
            geojson_dir.mkdir(exist_ok=True)
            
            service_area_geojson = geojson_dir / f"{prefix}_service_areas_{timestamp}.geojson"
            result_gdf.to_file(service_area_geojson, driver='GeoJSON')
            saved_files['service_areas_geojson'] = str(service_area_geojson)
            print(f"√ 服务区GeoJSON: {service_area_geojson}")
            
            # 3. 保存表格数据（CSV/Excel）
            print("\n保存表格数据...")
            tables_dir = self.config.output_dir / "tables"
            tables_dir.mkdir(exist_ok=True)
            
            # 3.1 便利度统计表
            if 'convenience_score' in result_gdf.columns:
                stats_cols = ['name', 'convenience_score', 'convenience_level', 'total_poi']
                
                # 添加维度得分
                for dim in ['supply_dimension', 'diversity_dimension', 'accessibility_dimension']:
                    if dim in result_gdf.columns:
                        stats_cols.append(dim)
                
                # 添加POI类别计数
                count_cols = [col for col in result_gdf.columns if col.startswith('count_')]
                stats_cols.extend(count_cols)
                
                # 确保列存在
                available_cols = [col for col in stats_cols if col in result_gdf.columns]
                stats_df = result_gdf[available_cols].copy()
                
                # 保存为CSV
                stats_csv = tables_dir / f"{prefix}_convenience_stats_{timestamp}.csv"
                stats_df.to_csv(stats_csv, index=False, encoding='utf-8-sig')
                saved_files['convenience_stats_csv'] = str(stats_csv)
                print(f"√ 便利度统计CSV: {stats_csv}")
                
                # 保存为Excel
                try:
                    stats_excel = tables_dir / f"{prefix}_convenience_stats_{timestamp}.xlsx"
                    stats_df.to_excel(stats_excel, index=False)
                    saved_files['convenience_stats_excel'] = str(stats_excel)
                    print(f"√ 便利度统计Excel: {stats_excel}")
                except Exception as e:
                    print(f"! 保存Excel失败: {e}")
            
            # 3.2 原始小区数据
            communities_csv = tables_dir / f"{prefix}_communities_{timestamp}.csv"
            # 移除几何列以保存为CSV
            communities_df = communities_gdf.copy()
            if 'geometry' in communities_df.columns:
                communities_df = communities_df.drop(columns=['geometry'])
            communities_df.to_csv(communities_csv, index=False, encoding='utf-8-sig')
            saved_files['communities_csv'] = str(communities_csv)
            print(f"√ 小区数据CSV: {communities_csv}")
            
            # 4. 保存JSON格式（用于程序读取）
            print("\n保存JSON数据...")
            json_dir = self.config.output_dir / "json"
            json_dir.mkdir(exist_ok=True)
            
            # 保存结果摘要
            summary = {
                'analysis_time': timestamp,
                'total_communities': len(result_gdf),
                'convenience_stats': {}
            }
            
            if 'convenience_score' in result_gdf.columns:
                summary['convenience_stats'] = {
                    'mean': float(result_gdf['convenience_score'].mean()),
                    'std': float(result_gdf['convenience_score'].std()),
                    'min': float(result_gdf['convenience_score'].min()),
                    'max': float(result_gdf['convenience_score'].max()),
                    'median': float(result_gdf['convenience_score'].median())
                }
            
            summary_json = json_dir / f"{prefix}_summary_{timestamp}.json"
            with open(summary_json, 'w', encoding='utf-8') as f:
                json.dump(summary, f, ensure_ascii=False, indent=2)
            saved_files['summary_json'] = str(summary_json)
            print(f"√ 分析摘要JSON: {summary_json}")
            
            print("\n☑ 所有数据导出完成")
            return saved_files
            
        except Exception as e:
            print(f"X 数据导出失败: {e}")
            import traceback
            traceback.print_exc()
            return saved_files
    
    def generate_report(self, result_gdf, visualizations, saved_files, analysis_time, weights_dict):
        """
        生成分析报告
        
        参数:
            result_gdf: 分析结果数据
            visualizations: 可视化文件路径
            saved_files: 保存的数据文件路径
            analysis_time: 分析时间
            weights_dict: CRITIC法权重字典
            
        返回:
            str: 报告文件路径
        """
        print("\n" + "=" * 60)
        print("步骤: 生成分析报告")
        print("-" * 60)
        
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            report_path = self.config.reports_dir / f"analysis_report_{timestamp}.txt"
            
            with open(report_path, 'w', encoding='utf-8') as f:
                f.write("=" * 70 + "\n")
                f.write("社区生活圈便利度评价分析报告（基于CRITIC法）\n")
                f.write("=" * 70 + "\n\n")
                
                f.write(f"报告生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"分析完成时间: {analysis_time}\n")
                f.write(f"研究区域: {self.config.target_city}-{self.config.target_district}\n")
                f.write(f"研究方法: 缓冲区({self.config.buffer_distance}米)+路网限制 + CRITIC客观赋权法\n\n")
                
                f.write("一、分析概况\n")
                f.write("-" * 40 + "\n")
                f.write(f"分析小区数量: {len(result_gdf)}\n")
                f.write(f"服务区半径: {self.config.buffer_distance} 米\n")
                f.write(f"道路缓冲宽度: {self.config.road_buffer_width} 米\n\n")
                
                f.write("二、CRITIC法权重计算结果\n")
                f.write("-" * 40 + "\n")
                if weights_dict:
                    sorted_weights = sorted(weights_dict.items(), key=lambda x: x[1], reverse=True)
                    for poi_type, weight in sorted_weights:
                        f.write(f"{poi_type}: {weight:.4f}\n")
                f.write("\n")
                
                f.write("三、便利度评价结果\n")
                f.write("-" * 40 + "\n")
                if 'convenience_score' in result_gdf.columns:
                    scores = result_gdf['convenience_score']
                    f.write(f"综合便利度得分统计:\n")
                    f.write(f"平均值: {scores.mean():.4f}\n")
                    f.write(f"标准差: {scores.std():.4f}\n")
                    f.write(f"最小值: {scores.min():.4f}\n")
                    f.write(f"最大值: {scores.max():.4f}\n")
                    f.write(f"中位数: {scores.median():.4f}\n\n")
                
                if 'convenience_level' in result_gdf.columns:
                    f.write("便利度等级分布:\n")
                    level_counts = result_gdf['convenience_level'].value_counts().sort_index()
                    for level, count in level_counts.items():
                        percentage = count / len(result_gdf) * 100
                        f.write(f"{level}: {count}个 ({percentage:.1f}%)\n")
                    f.write("\n")
                
                # POI统计
                if 'total_poi' in result_gdf.columns:
                    f.write("POI设施统计:\n")
                    f.write(f"POI总数: {int(result_gdf['total_poi'].sum())}\n")
                    f.write(f"平均每个服务区POI数: {result_gdf['total_poi'].mean():.1f}\n")
                    if 'poi_density' in result_gdf.columns:
                        f.write(f"平均POI密度: {result_gdf['poi_density'].mean():.1f} 个/平方公里\n")
                    f.write("\n")
                
                f.write("四、便利度最高的小区（TOP 10）\n")
                f.write("-" * 40 + "\n")
                if 'convenience_score' in result_gdf.columns and 'name' in result_gdf.columns:
                    top_10 = result_gdf.nlargest(10, 'convenience_score')
                    for i, (idx, row) in enumerate(top_10.iterrows(), 1):
                        level = row.get('convenience_level', '未知')
                        f.write(f"{i:2d}. {row.get('name', f'小区{idx}')}\n")
                        f.write(f"    得分: {row['convenience_score']:.3f} ({level})\n")
                        if 'total_poi' in row:
                            f.write(f"    POI数量: {int(row['total_poi'])}\n")
                        f.write("\n")
                
                f.write("五、生成文件清单\n")
                f.write("-" * 40 + "\n")
                # 数据文件
                f.write("1. 数据文件:\n")
                for file_type, file_path in saved_files.items():
                    if isinstance(file_path, str):
                        file_name = Path(file_path).name
                        f.write(f"  - {file_name}\n")
                
                # 可视化文件
                f.write("\n2. 可视化图表:\n")
                if visualizations:
                    for viz_type, viz_path in visualizations.items():
                        if isinstance(viz_path, str):
                            file_name = Path(viz_path).name
                            f.write(f"  - {file_name}\n")
                        elif isinstance(viz_path, list):
                            for path in viz_path:
                                file_name = Path(path).name
                                f.write(f"  - {file_name}\n")
                
                f.write("\n六、研究方法说明\n")
                f.write("-" * 40 + "\n")
                f.write("本研究采用'缓冲区+路网限制'的方法进行社区生活圈便利度评价:\n")
                f.write("1. 为每个小区生成基于真实路网的服务区，而非简单圆形缓冲区\n")
                f.write("2. 统计服务区内各类POI设施数量\n")
                f.write("3. 使用CRITIC客观赋权法计算指标权重\n")
                f.write("4. 从供给、多样性、可达性三个维度综合评价\n")
                f.write("5. 计算综合便利度得分并进行分级\n\n")
                
                f.write("七、优化建议\n")
                f.write("-" * 40 + "\n")
                f.write("1. 对低便利度小区，优先补充教育、医疗等基本公共服务设施\n")
                f.write("2. 优化交通网络，提高边缘小区与中心设施的连接性\n")
                f.write("3. 在新建住宅区同步规划15分钟生活圈配套设施\n")
                f.write("4. 建立动态监测机制，定期评估社区生活圈便利度\n")
                f.write("5. 鼓励混合土地利用，提高设施使用效率\n\n")
                
                f.write("=" * 70 + "\n")
                f.write("报告结束\n")
                f.write("=" * 70 + "\n")
            
            print(f"√ 分析报告已生成: {report_path}")
            return str(report_path)
            
        except Exception as e:
            print(f"X 报告生成失败: {e}")
            import traceback
            traceback.print_exc()
            return None

# 8.2 创建数据导出器实例
data_exporter = DataExporter(config)
print("☑ 数据导出器初始化完成")

☑ 数据导出器初始化完成


In [11]:
# 9. 主程序执行函数（优化版，集成所有模块）
def main_analysis_pipeline():
    """
    完整的主程序执行流程 - 优化版本
    集成了POI采集、可视化、数据导出等所有模块
    修复了所有已知问题，确保流程完整性
    """
    print("=" * 80)
    print("社区生活圈便利度评价系统 - 完整执行流程")
    print("研究方法: 缓冲区+路网限制 + CRITIC客观赋权法")
    print("=" * 80)
    
    # 记录开始时间
    start_time = datetime.now()
    print(f"开始时间: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    try:
        # 1. 文件路径配置
        print("\n■ 步骤1: 配置文件路径")
        print("-" * 40)
        
        # 使用配置中的用户文件路径
        file_config = {
            'boundary': config.user_file_paths['boundary'],
            'road_network': config.user_file_paths['road_network'],
            'poi_local': config.user_file_paths['poi_local'],
            'community': config.user_file_paths['community']
        }
        
        # 检查文件是否存在
        missing_files = []
        for file_type, file_path in file_config.items():
            file_path_obj = Path(file_path)
            if file_path_obj.exists():
                print(f"√ {file_type}: {file_path}")
            else:
                print(f"! {file_type}: 文件不存在 - {file_path}")
                missing_files.append(file_type)
        
        # 如果POI文件不存在，尝试自动采集
        if 'poi_local' in missing_files:
            print("\n! POI文件不存在，尝试自动采集...")
            poi_output_path = config.data_dir / "collected_poi.geojson"
            
            # 测试API连接
            if test_api_connection():
                print("√ API连接正常，开始采集POI数据...")
                poi_gdf = poi_collector.batch_collect(
                    save_path=poi_output_path,
                    verbose=True
                )
                if not poi_gdf.empty:
                    file_config['poi_local'] = str(poi_output_path)
                    print(f"√ POI采集完成，已保存到: {poi_output_path}")
                    missing_files.remove('poi_local')
                else:
                    print("X POI采集失败，请检查API密钥或网络连接")
            else:
                print("X API连接失败，无法采集POI数据")
        
        # 检查是否还有缺失的必要文件
        essential_files = ['boundary', 'road_network', 'community']
        missing_essential = [f for f in essential_files if f in missing_files]
        
        if missing_essential:
            print(f"\nX 缺少必要文件: {', '.join(missing_essential)}")
            print("请修改Config类中的user_file_paths配置后重新运行")
            
            # 计算耗时
            end_time = datetime.now()
            total_time = (end_time - start_time).total_seconds()
            print(f"\n总运行时间: {total_time:.1f}秒")
            return None
        
        print("√ 所有必要文件检查通过")
        
        # 2. 初始化空间分析器
        print("\n■ 步骤2: 初始化分析器")
        print("-" * 40)
        spatial_analyzer = SpatialAnalyzer(config)
        print("√ 空间分析器初始化完成")
        
        # 3. 加载空间数据
        print("\n■ 步骤3: 加载空间数据")
        print("-" * 40)
        boundary, roads, poi, communities = spatial_analyzer.load_and_unify_data(
            file_config['boundary'],
            file_config['road_network'],
            file_config['poi_local'],
            file_config['community']
        )
        
        if any(x is None for x in [boundary, roads, poi, communities]):
            print("X 空间数据加载失败")
            
            # 计算耗时
            end_time = datetime.now()
            total_time = (end_time - start_time).total_seconds()
            print(f"\n总运行时间: {total_time:.1f}秒")
            return None
        
        print(f"√ 空间数据加载成功")
        print(f"小区数量: {len(communities)}")
        print(f"POI数量: {len(poi)}")
        
        # 4. 生成路网服务区
        print("\n■ 步骤4: 生成路网服务区")
        print("-" * 40)
        service_areas = spatial_analyzer.generate_road_network_service_areas(
            communities, roads,
            distance_meters=config.buffer_distance
        )
        
        if service_areas.empty:
            print("X 服务区生成失败")
            
            # 计算耗时
            end_time = datetime.now()
            total_time = (end_time - start_time).total_seconds()
            print(f"\n总运行时间: {total_time:.1f}秒")
            return None
        
        print(f"√ 成功生成 {len(service_areas)} 个服务区")
        
        # 5. 统计服务区内POI
        print("\n■ 步骤5: 统计服务区内POI")
        print("-" * 40)
        result_with_poi = spatial_analyzer.count_poi_in_service_areas(service_areas, poi)
        
        # 检查POI统计结果
        count_cols = [col for col in result_with_poi.columns if col.startswith('count_')]
        if count_cols:
            print(f"√ POI统计完成，共识别 {len(count_cols)} 类POI")
        else:
            print("! 未识别到POI类别，便利度计算可能不准确")
        
        # 6. 计算便利度（使用CRITIC法）
        print("\n■ 步骤6: 计算便利度（CRITIC法）")
        print("-" * 40)
        
        # 6.1 使用CRITIC法计算权重
        critic_calculator = CRITICConvenienceCalculator(config)  # 初始化计算器
        weights = critic_calculator.calculate_critic_weights(result_with_poi)
        
        # 6.2 计算综合便利度得分
        result_with_score = critic_calculator.calculate_convenience_score(result_with_poi, weights)
        
        # 6.3 获取TOP/BOTTOM小区
        top_df, bottom_df = critic_calculator.get_top_bottom_communities(result_with_score)
        if not top_df.empty:
            print(f"\n☑ 便利度最高的5个小区:")
            print(top_df.to_string(index=False))
        
        #6.4 权重敏感性分析
        result_with_score, sensitivity_correlations = sensitivity_analyzer.perform_sensitivity_analysis(
            result_with_score, 
            original_score_col='convenience_score',
            verbose=True
        )
        # 7. 生成可视化图表
        print("\n■ 步骤7: 生成可视化图表")
        print("-" * 40)
        visualizations = visualizer.generate_all_visualizations(
            boundary, roads, communities, result_with_score, result_with_score
        )
        
        # 8. 保存分析结果
        print("\n■ 步骤8: 保存分析结果")
        print("-" * 40)
        saved_files = data_exporter.save_analysis_results(result_with_score, communities)
        
        # 9. 生成分析报告
        print("\n■ 步骤9: 生成分析报告")
        print("-" * 40)
        analysis_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        report_path = data_exporter.generate_report(
            result_with_score, visualizations, saved_files, analysis_time, weights
        )
        
        # 计算总耗时
        end_time = datetime.now()
        total_time = (end_time - start_time).total_seconds()
        
        print("\n" + "=" * 80)
        print("☑ 分析流程全部完成!")
        print("=" * 80)
        
        print(f"\n分析结果摘要:")
        print(f"分析小区数量: {len(communities)}")
        print(f"成功生成服务区: {len(service_areas)}")
        
        if 'convenience_score' in result_with_score.columns:
            scores = result_with_score['convenience_score']
            print(f"平均便利度: {scores.mean():.3f}")
            print(f"最高得分: {scores.max():.3f}")
            print(f"最低得分: {scores.min():.3f}")
        
        if 'convenience_level' in result_with_score.columns:
            high_count = len(result_with_score[result_with_score['convenience_level'] == '高便利度'])
            low_count = len(result_with_score[result_with_score['convenience_level'] == '低便利度'])
            print(f"高便利度小区: {high_count}个")
            print(f"低便利度小区: {low_count}个")
        
        print(f"\n总耗时: {total_time:.1f}秒 ({total_time/60:.1f}分钟)")
        print(f"开始时间: {start_time.strftime('%H:%M:%S')}")
        print(f"结束时间: {end_time.strftime('%H:%M:%S')}")
        
        if report_path:
            print(f"\n输出目录: {config.output_dir}")
            print(f"分析报告: {report_path}")
        
        print("\n" + "=" * 80)
        
        return {
            'boundary': boundary,
            'roads': roads,
            'poi': poi,
            'communities': communities,
            'service_areas': service_areas,
            'result': result_with_score,
            'weights': weights,
            'visualizations': visualizations,
            'saved_files': saved_files,
            'report_path': report_path,
            'processing_time': total_time
        }
        
    except Exception as e:
        print(f"\nX 分析过程中发生错误: {e}")
        import traceback
        traceback.print_exc()
        
        # 关键修复: 即使在异常情况下也计算耗时
        end_time = datetime.now()
        total_time = (end_time - start_time).total_seconds()
        print(f"\n总运行时间: {total_time:.1f}秒")
        return None

# 10. 快捷运行函数
def quick_run():
    """
    快捷运行函数 - 一键执行完整分析流程
    """
    print("正在启动完整分析流程...")
    results = main_analysis_pipeline()
    
    if results:
        print("\n" + "=" * 60)
        print("☑ 分析成功完成!")
        print("=" * 60)
        print(f"详细结果已保存到: {results.get('report_path', '未知')}")
        
        # 显示便利度最高的3个小区
        if 'result' in results:
            result_df = results['result']
            if 'convenience_score' in result_df.columns:
                top_3 = result_df.nlargest(3, 'convenience_score')
                print("\n便利度最高的3个小区:")
                for i, (idx, row) in enumerate(top_3.iterrows(), 1):
                    print(f"  {i}. {row.get('name', f'小区{idx}')} - 得分: {row['convenience_score']:.3f}")
    else:
        print("\n" + "=" * 60)
        print("X 分析失败，请检查错误信息")
        print("=" * 60)
    
    return results

In [12]:
# 11. 主程序入口
if __name__ == "__main__":
    """
    主程序入口
    提供交互式菜单选择运行模式
    """
    print("\n" + "=" * 60)
    print("社区生活圈便利度评价系统")
    print("=" * 60)
    
    print("\n请选择运行模式:")
    print("1. 完整分析流程（推荐）")
    print("2. 仅测试数据加载")
    print("3. 测试API连接")
    print("4. 仅采集POI数据")
    print("5. 退出")
    
    try:
        choice = input("\n请输入选择(1-5): ").strip()
        
        if choice == '1':
            print("\n启动完整分析流程...")
            results = quick_run()
        elif choice == '2':
            print("\n测试数据加载...")
            # 使用配置中的路径
            test_config = config.user_file_paths
            
            # 测试文件是否存在
            for file_type, file_path in test_config.items():
                file_path_obj = Path(file_path)
                if file_path_obj.exists():
                    print(f"√ {file_type}: {file_path}")
                    
                    # 如果是小区数据，尝试加载预览
                    if file_type == 'community' and file_path.endswith(('.csv', '.xlsx', '.xls')):
                        try:
                            processor = CSVCommunityProcessor(file_path)
                            df = processor.load_and_validate()
                            print(f"数据形状: {df.shape}")
                            print(f"前3行预览:")
                            print(df.head(3).to_string())
                        except Exception as e:
                            print(f"! 加载失败: {e}")
                else:
                    print(f"X {file_type}: 文件不存在 - {file_path}")
            
            print("\n测试完成")
        elif choice == '3':
            print("\n测试API连接...")
            test_api_connection()
        elif choice == '4':
            print("\n仅采集POI数据...")
            # 指定保存路径
            poi_output_path = config.data_dir / "collected_poi.geojson"
            print(f"POI将保存到: {poi_output_path}")
            
            # 采集POI
            poi_gdf = poi_collector.batch_collect(
                save_path=poi_output_path,
                verbose=True
            )
            
            if not poi_gdf.empty:
                print(f"\n采集完成!")
                print(f"采集到 {len(poi_gdf)} 个POI")
                print(f"已保存到: {poi_output_path}")
            else:
                print("\n采集失败!")
        elif choice == '5':
            print("\n退出程序")
        else:
            print("\n无效选择")
            
    except KeyboardInterrupt:
        print("\n\n用户取消了操作")
    except Exception as e:
        print(f"\nX 运行错误: {e}")

# 12. 使用说明
print("\n" + "=" * 60)
print("使用说明:")
print("=" * 60)
print("1. 首先修改Config类中的配置:")
print("   - 将AMAP_KEY替换为您的高德地图API密钥")
print("   - 检查user_file_paths中的文件路径是否正确")
print("\n2. 运行以下代码开始分析:")
print("   results = quick_run()  # 一键运行完整分析")
print("\n3. 或者运行主菜单:")
print("   在Jupyter中运行最后一个代码块，选择模式1")
print("=" * 60)


社区生活圈便利度评价系统

请选择运行模式:
1. 完整分析流程（推荐）
2. 仅测试数据加载
3. 测试API连接
4. 仅采集POI数据
5. 退出

请输入选择(1-5): 1

启动完整分析流程...
正在启动完整分析流程...
社区生活圈便利度评价系统 - 完整执行流程
研究方法: 缓冲区+路网限制 + CRITIC客观赋权法
开始时间: 2026-04-22 22:08:51

■ 步骤1: 配置文件路径
----------------------------------------
√ boundary: C:\Users\33353\Desktop\作业\441302.shp
√ road_network: D:\惠城区道路路网_441302_Shapefile_(poi86.com)\441302.shp
√ poi_local: D:/Jupyter notebook 代码/data/collected_poi.geojson
√ community: D:/Jupyter notebook 代码/惠城区小区数据/惠城区居民小区_20260317_043627.csv
√ 所有必要文件检查通过

■ 步骤2: 初始化分析器
----------------------------------------
√ 空间分析器初始化完成

■ 步骤3: 加载空间数据
----------------------------------------
步骤1: 加载并统一空间数据
------------------------------------------------------------
加载行政区边界...
  原始CRS: EPSG:4326，要素数: 19
加载路网数据...
  ! 路网数据缺少CRS，尝试自动检测...
  已设置CRS为: EPSG:4326
  原始CRS: EPSG:4326，线段数: 9783
加载POI数据...
  POI数据CRS: EPSG:4326，点数: 6372
加载小区数据...
加载数据文件: D:\Jupyter notebook 代码\惠城区小区数据\惠城区居民小区_20260317_043627.csv
√ 使用编码 utf-8-sig 成功读取CSV文件
数据形状: (220, 18

生成服务区:   0%|          | 0/220 [00:00<?, ?it/s]


☑ 服务区生成完成
------------------------------------------------------------
成功生成: 220个服务区
生成失败: 0个
处理时间: 75.6秒
平均时间: 0.34秒/个

面积统计:
  平均面积: 2,804,149平方米
  最大面积: 3,313,451平方米
  最小面积: 71,786平方米
  总面积: 616,912,881平方米(616.9平方公里)
√ 成功生成 220 个服务区

■ 步骤5: 统计服务区内POI
----------------------------------------

步骤3: 统计服务区内POI（基于CSV分类字段）
------------------------------------------------------------
执行空间连接(点在面内分析)...
  空间连接成功: 23903条连接记录
对重复POI进行去重...
  去重依据: 无
  去重前记录数: 23903
  去重后记录数: 23903
  去除重复: 0条
按CSV分类字段统计POI数量...

☑ POI统计摘要（基于CSV分类字段）:
  总POI数: 23903
  平均每个服务区POI数: 108.7
  平均POI密度: 37.3个/平方公里

  各类POI数量:
  交通设施_停车场: 1068
  交通设施_公交站: 865
  交通设施_加油站: 189
  交通设施_汽车站: 20
  交通设施_火车站: 22
  医疗设施_医院: 737
  医疗设施_卫生服务中心: 513
  医疗设施_卫生院: 272
  医疗设施_疾控中心: 67
  医疗设施_药店: 990
  医疗设施_诊所: 635
  商业设施_便利店: 1037
  商业设施_商场: 658
  商业设施_市场: 538
  商业设施_超市: 574
  商业设施_银行: 1169
  商业设施_餐厅: 1027
  商业设施_餐饮: 883
  商业设施_饭店: 621
  教育设施_中学: 474
  教育设施_培训机构: 1322
  教育设施_大学: 31
  教育设施_学校: 528
  教育设施_小学: 406
  教育设施_幼儿园: 663
  教育设施